# Analytical workflow explained from repository scripts

**Manuscript:** *Contamination and taxon sampling explain conflicting eukaryote placements*

This notebook is a **reproducibility guide**. It follows the manuscript order and embeds the **complete example scripts** used in the repository (with placeholder paths). Numerical results are defined by deposited tables and trees under `data/`; scripts under `scripts/` are templates that document how those products were generated.

| Part | Scientific question | Main displays | Key `data/` |
| --- | --- | --- | --- |
| **A — Contamination landscape** | Is contamination random, or concentrated in lineages relevant to eukaryogenesis? | **Fig. 1**, Extended Data contamination figures | `data/decontamination/assessment/` |
| **B — Phylogenomics** | After controlling contamination and sampling imbalance, where do eukaryotes sit relative to Asgard and TACK? | **Fig. 2–3**, AU, CAT-GTR, post-hoc audit | `data/genome_sets/`, `data/PMS/`, `data/alignments/`, `data/trees/` |
| **C — ESP inventories** | Does the same contamination bias also inflate ESP counts? | **Fig. 4** | `data/ESP/` |

**Repository:** https://github.com/tjcadd2020/Asgard-Eukaryote-Phylogenomics-2026

### How to use this notebook
1. Read the markdown cells for scientific logic (why each step exists).
2. Use the full scripts as templates: replace `/path/to/...` with local paths.
3. Prefer deposited `data/` files when verifying published numbers; re-running every HPC job is optional.


## Shared genome accession tables

These lists define **which genomes** enter the study (contamination survey and phylogenomics share accessions; analytical goals differ).

| Genome set | Role | Documented in |
| --- | --- | --- |
| GS-Zhang2025 | Imbalanced benchmark (Zhang et al. 2025) | Supplementary Table 2 · `data/genome_sets/` |
| GS-Liu2021 | Imbalanced benchmark (Liu et al. 2021) | Supplementary Table 3 |
| GS-Zhang2025-B | Balanced version of GS-Zhang2025 | Supplementary Table 4 |
| GS-Liu2021-B | Balanced version of GS-Liu2021 | Supplementary Table 5 |
| GS-Present-B | Independently assembled balanced set (this study) | Supplementary Table 6 |

Assemblies were obtained from NCBI using these accessions. Full raw FASTA collections are not re-hosted in the repository.


---
# Part A — Contamination landscape across archaeal MAGs

**Results block:** “Key lineages harbour more contamination”.

**Logic**
1. Detect candidate exogenous sequences in MAGs (CAT + geNomad).
2. Summarise genome-level frequencies across major archaeal groups (**Fig. 1a**).
3. Zoom into Asgard class/order patterns and contaminant-derived eukaryote-like proteins (**Fig. 1b–c**).

Part A does **not** place eukaryotes on the tree. It establishes that contamination is pervasive, uneven, and enriched in the same Asgard lineages often proposed as close relatives of eukaryotes.


## A1 — Identification of candidate exogenous sequences

### Purpose
Flag contigs that are unresolved or non-archaeal so they can be (i) counted for the contamination survey and (ii) removed before phylogenomic marker extraction.

### Methods criteria (CAT)
Contig-level classifications with CAT v6.0 (sensitive). Exclude “no ORFs found”. Flag as exogenous if:
1. classification is root-only; or
2. second rank is not cellular organisms (NCBI taxid **131567**); or
3. third rank is not Archaea (taxid **2157**).

### Viral detection (geNomad)
geNomad v1.8.0, **conservative** mode. Free-virus calls (`topology != Provirus`) inform removal; **proviruses are retained**. Compared with Phager, geNomad gave fewer lineage-inconsistent calls with stronger hallmark support and was used as the conservative detector (**Extended Data Fig. 1**).

### Workflow order
`run_decontamination.sh` runs CAT + geNomad per genome → `flag_exogenous_contigs_CAT.R` → `collect_genomad_virus_summaries.R` → `remove_exogenous_and_extract_markers.R` (union of flags, then cleaned marker proteins).


### Full script: `scripts/decontamination/run_decontamination.sh`

```bash
#!/bin/bash
# ============================================================
# Script : run_decontamination.sh (EXAMPLE TEMPLATE)
# Purpose: Identify candidate exogenous sequences in archaeal MAGs
#          and generate decontaminated (clean) genome collections.
#
# Manuscript
# ----------
# Contamination and taxon sampling explain conflicting eukaryote
# placements
#
# Methods section mirrored by this script
# ---------------------------------------
# "Identification of candidate exogenous sequences"
#
# Contig-level taxonomic classifications were generated with CAT v6.0
# (sensitive mode). Contigs annotated as "no ORFs found" were excluded.
# Candidate exogenous contigs were defined as those for which:
#   (i)   classification was restricted to the root level; or
#   (ii)  the second taxonomic rank was not cellular organisms
#         (NCBI Taxonomy ID 131567); or
#   (iii) the third taxonomic rank was not Archaea
#         (NCBI Taxonomy ID 2157).
# Contaminant origins were assigned to bacterial, eukaryotic, viral or
# unclassified categories. Viral sequences were additionally identified
# with geNomad v1.8.0 (conservative mode). Free-virus calls
# (topology != Provirus) were retained; proviruses were not treated as
# exogenous contamination. Contigs flagged by CAT and/or geNomad were
# removed to produce clean genome collections. Genomes before and after
# contaminant removal are referred to as raw and clean datasets.
#
# Companion R scripts (same repository)
# --------------------------------------
#   scripts/decontamination/flag_exogenous_contigs_CAT.R
#       → parse CAT official-names files; apply criteria (i)–(iii)
#   scripts/decontamination/collect_genomad_virus_summaries.R
#       → aggregate *_virus_summary.tsv tables
#   scripts/decontamination/remove_exogenous_and_extract_markers.R
#       → union(CAT, geNomad free-virus) and extract cleaned markers
#
# Tools  : CAT v6.0, geNomad v1.8.0, R
# Stage  : Primary decontamination
# Note   : EXAMPLE template only. Paths are placeholders and must be
#          adapted to the local environment.
# ============================================================

set -euo pipefail

# ------------------ Configuration (MODIFY AS NEEDED) ------------------
WORK_DIR="/path/to/your/project"
SOFTWARE_DIR="/path/to/your/software"
SCRIPT_DIR="${WORK_DIR}/scripts/decontamination"

RAW_GENOMES_DIR="${WORK_DIR}/data/genomes/raw_MAGs"
OUTPUT_DIR="${WORK_DIR}/results/decontamination"

CAT_DB="${SOFTWARE_DIR}/20240422_CAT_nr/db"
CAT_TAX="${SOFTWARE_DIR}/20240422_CAT_nr/tax"
GENOMAD_DB="${SOFTWARE_DIR}/genomad_db"

THREADS_CAT=18
THREADS_GENOMAD=4

mkdir -p \
  "${OUTPUT_DIR}/cat_classification/contigs" \
  "${OUTPUT_DIR}/cat_classification/add_names" \
  "${OUTPUT_DIR}/cat_classification/summaries" \
  "${OUTPUT_DIR}/genomad" \
  "${OUTPUT_DIR}/flags" \
  "${OUTPUT_DIR}/clean_genomes" \
  "${OUTPUT_DIR}/logs"

echo "=== Primary decontamination workflow ==="
echo "Input raw genomes : ${RAW_GENOMES_DIR}"
echo "Output directory  : ${OUTPUT_DIR}"
date
echo ""

# ============================================================
# Helper: process a single genome (CAT + geNomad)
# ============================================================
process_genome() {
  local genome_fna="$1"
  local base
  base=$(basename "${genome_fna}" .fna)

  echo ">>> Processing ${base}"

  # ----------------------------------------------------------
  # 1. Contig-level taxonomic classification (CAT v6.0, sensitive)
  # ----------------------------------------------------------
  echo "    CAT classification (sensitive mode)..."
  CAT_pack contigs \
    -c "${genome_fna}" \
    -d "${CAT_DB}" \
    -t "${CAT_TAX}" \
    -o "${OUTPUT_DIR}/cat_classification/contigs/${base}" \
    --sensitive \
    -n "${THREADS_CAT}" \
    > "${OUTPUT_DIR}/logs/${base}.CAT_contigs.log" 2>&1

  # Official names file is the input to flag_exogenous_contigs_CAT.R
  CAT_pack add_names \
    -i "${OUTPUT_DIR}/cat_classification/contigs/${base}.contig2classification.txt" \
    -o "${OUTPUT_DIR}/cat_classification/add_names/${base}.contig2classification.official_names.txt" \
    -t "${CAT_TAX}" \
    --only_official \
    > "${OUTPUT_DIR}/logs/${base}.CAT_add_names.log" 2>&1

  CAT_pack summarise \
    -c "${genome_fna}" \
    -i "${OUTPUT_DIR}/cat_classification/contigs/${base}.contig2classification.txt" \
    -o "${OUTPUT_DIR}/cat_classification/summaries/${base}.summary.txt" \
    > "${OUTPUT_DIR}/logs/${base}.CAT_summarise.log" 2>&1

  # ----------------------------------------------------------
  # 2. Independent viral detection (geNomad v1.8.0, conservative)
  # ----------------------------------------------------------
  echo "    geNomad viral detection (conservative mode)..."
  genomad end-to-end \
    --threads "${THREADS_GENOMAD}" \
    --conservative \
    "${genome_fna}" \
    "${OUTPUT_DIR}/genomad" \
    "${GENOMAD_DB}" \
    > "${OUTPUT_DIR}/logs/${base}.genomad.log" 2>&1

  echo "    Done: ${base}"
  echo ""
}

# ============================================================
# Batch: CAT + geNomad for all genomes
# ============================================================
echo ">>> Scanning input directory for *.fna files..."
shopt -s nullglob
genome_files=("${RAW_GENOMES_DIR}"/*.fna)

if [[ ${#genome_files[@]} -eq 0 ]]; then
  echo "WARNING: No .fna files found in ${RAW_GENOMES_DIR}"
  echo "         Place genomes there or call process_genome on a single file."
else
  for fna in "${genome_files[@]}"; do
    process_genome "${fna}"
  done
fi

# ============================================================
# 3. Flag CAT-based exogenous contigs (R)
#    Criteria (i)–(iii); exclude "no ORFs found"
#    Input : *.contig2classification.official_names.txt
#    Output: candidate_exogenous_contigs_CAT.txt
# ============================================================
echo ">>> Flagging CAT exogenous contigs..."
(
  cd "${OUTPUT_DIR}/cat_classification/add_names"
  Rscript "${SCRIPT_DIR}/flag_exogenous_contigs_CAT.R"
  mv -f candidate_exogenous_contigs_CAT.txt \
    "${OUTPUT_DIR}/flags/candidate_exogenous_contigs_CAT.txt"
)

# ============================================================
# 4. Aggregate geNomad virus summaries (R)
#    Input : recursive *_virus_summary.tsv under OUTPUT_DIR/genomad
#    Output: genomad_conservative.txt
# ============================================================
echo ">>> Collecting geNomad virus summaries..."
(
  cd "${OUTPUT_DIR}/genomad"
  Rscript "${SCRIPT_DIR}/collect_genomad_virus_summaries.R"
  mv -f genomad_conservative.txt \
    "${OUTPUT_DIR}/flags/genomad_conservative.txt"
)

# ============================================================
# 5. Merge flags and write clean genomes / cleaned markers (R)
#    union(CAT exogenous, geNomad free-virus [topology != Provirus])
#    Then remove flagged contigs from annotations / genomes and
#    extract representative eggNOG proteins for PMS construction.
#
#    See: remove_exogenous_and_extract_markers.R
# ============================================================
echo ">>> Removing exogenous contigs and extracting cleaned markers..."
# Rscript "${SCRIPT_DIR}/remove_exogenous_and_extract_markers.R"
# (Paths inside the R script must point to the flag tables above
#  and to the relevant genome / annotation collections.)

echo ""
echo "=== Primary decontamination workflow finished ==="
date
echo "Flag tables : ${OUTPUT_DIR}/flags/"
echo "Clean genomes / cleaned marker FASTAs are produced by the"
echo "companion R script remove_exogenous_and_extract_markers.R"
echo "Raw vs clean designations correspond to genome sets before and"
echo "after contaminant removal, as defined in the manuscript."
```


### Full script: `scripts/decontamination/flag_exogenous_contigs_CAT.R`

```r
################################################################################
# Flag candidate exogenous contigs from CAT official-names output
#
# Manuscript
# ----------
# Contamination and taxon sampling explain conflicting eukaryote placements
#
# Methods criteria (Identification of candidate exogenous sequences)
# -----------------------------------------------------------------
# Contig-level taxonomic classifications were generated with CAT v6.0
# (sensitive mode). Contigs annotated as "no ORFs found" were excluded.
# Candidate exogenous contigs were defined as those for which:
#   (i)   classification was restricted to the root level; or
#   (ii)  the second taxonomic rank was not cellular organisms
#         (NCBI Taxonomy ID 131567); or
#   (iii) the third taxonomic rank was not Archaea
#         (NCBI Taxonomy ID 2157).
#
# Input
# -----
# CAT_pack add_names output files matching:
#   *.contig2classification.official_names.txt
#
# Output
# ------
# Tab-delimited table of flagged contigs with genome identifiers
################################################################################

# Working directory should contain the CAT official-names files
class_files <- dir(pattern = "\\.contig2classification\\.official_names\\.txt$")

if (length(class_files) == 0) {
  stop("No *.contig2classification.official_names.txt files found in the working directory.")
}

cat_data <- data.frame()

for (x in class_files) {
  message("Processing: ", x)

  cat_name_2 <- read.delim(x, stringsAsFactors = FALSE, sep = "\t")

  # Exclude contigs with no predicted ORFs
  cat_name <- subset(cat_name_2, reason != "no ORFs found")

  if (nrow(cat_name) == 0) {
    next
  }

  # Parse lineage depth and rank taxids
  lineage_depth <- sapply(cat_name$lineage, function(y) {
    length(strsplit(y, ";")[[1]])
  })
  rank2 <- sapply(cat_name$lineage, function(y) {
    parts <- strsplit(y, ";")[[1]]
    if (length(parts) >= 2) parts[2] else NA_character_
  })
  rank3 <- sapply(cat_name$lineage, function(y) {
    parts <- strsplit(y, ";")[[1]]
    if (length(parts) >= 3) parts[3] else NA_character_
  })

  # Flag candidate exogenous contigs
  # (i)   root only          → lineage_depth == 1
  # (ii)  not cellular organisms (131567) at rank 2
  # (iii) not Archaea (2157) at rank 3
  is_exogenous <- (lineage_depth == 1) |
    (!is.na(rank2) & rank2 != "131567") |
    (!is.na(rank3) & rank3 != "2157")

  chimeric_contigs <- cat_name[is_exogenous, ]

  if (nrow(chimeric_contigs) > 0) {
    genome_id <- sub("\\.contig2classification\\.official_names\\.txt$", "", x)
    cat_data <- rbind(
      cat_data,
      data.frame(genome = genome_id, chimeric_contigs, stringsAsFactors = FALSE)
    )
  }
}

outfile <- "candidate_exogenous_contigs_CAT.txt"
write.table(
  cat_data,
  file = outfile,
  row.names = FALSE,
  quote = FALSE,
  sep = "\t"
)

message("Flagged contigs written to: ", outfile)
message("Total flagged contig records: ", nrow(cat_data))
################################################################################
```


### Full script: `scripts/decontamination/collect_genomad_virus_summaries.R`

```r
################################################################################
# Collect geNomad virus-summary tables across genomes
#
# Manuscript
# ----------
# Contamination and taxon sampling explain conflicting eukaryote placements
#
# Methods
# -------
# Viral sequences were independently identified using geNomad v1.8.0 with the
# conservative detection mode enabled. This script aggregates
# *_virus_summary.tsv outputs into a single table for downstream filtering.
#
# Input  : recursive search for *_virus_summary.tsv under the working directory
# Output : genomad_conservative.xlsx (or .txt)
################################################################################

genomad_files <- dir(pattern = "_virus_summary\\.tsv$", recursive = TRUE)

if (length(genomad_files) == 0) {
  stop("No *_virus_summary.tsv files found.")
}

genomad_data <- data.frame()

for (x in genomad_files) {
  message("Reading: ", x)
  genomad <- read.table(x, stringsAsFactors = FALSE, header = TRUE, sep = "\t")

  if (nrow(genomad) == 0) {
    next
  }

  # Genome ID = first path component (adjust if your directory layout differs)
  genome_id <- strsplit(x, "/")[[1]][1]
  genomad <- transform(genomad, genome = genome_id)
  genomad_data <- rbind(genomad_data, genomad)
}

# Prefer a plain TSV for portability; Excel optional
outfile <- "genomad_conservative.txt"
write.table(
  genomad_data,
  file = outfile,
  row.names = FALSE,
  quote = FALSE,
  sep = "\t"
)
message("Wrote ", nrow(genomad_data), " rows to ", outfile)

# Optional Excel export (requires Java for xlsx; prefer openxlsx or readxl workflows)
# if (requireNamespace("openxlsx", quietly = TRUE)) {
#   openxlsx::write.xlsx(genomad_data, "genomad_conservative.xlsx")
# }
################################################################################
```


### Full script: `scripts/decontamination/remove_exogenous_and_extract_markers.R`

```r
################################################################################
# Remove exogenous contigs and extract representative eggNOG marker proteins
#
# Manuscript
# ----------
# Contamination and taxon sampling explain conflicting eukaryote placements
#
# Methods mirrored by this script
# --------------------------------
# Primary decontamination combines:
#   (1) CAT-flagged candidate exogenous contigs (root / non-cellular /
#       non-Archaea ranks; "no ORFs found" already excluded upstream)
#   (2) geNomad conservative viral calls, excluding proviruses
#       (topology != "Provirus")
#
# Contigs flagged by either source are removed before orthologue selection.
# For each conserved eggNOG family, a single representative protein sequence
# is retained per genome after paralogue filtering, and written to
# per-family FASTA files used for single-protein trees and subsequent
# phylogenetic marker set (PMS) construction.
#
# Note
# ----
# Paths below are placeholders. Adapt to the local environment before running.
################################################################################

library(Biostrings)

# Prefer readxl / openxlsx over xlsx (no Java dependency)
if (!requireNamespace("readxl", quietly = TRUE)) {
  stop("Install readxl: install.packages('readxl')")
}
library(readxl)

##############################
# 1. Load geNomad virus calls
##############################
# Tables may be split by assembly level or data source; bind into one object.
genomad_contigs <- read_excel(
  "/path/to/genomad_conservative_MAG_archaea_contig.xlsx", sheet = 1
)
genomad_scaffolds <- read_excel(
  "/path/to/genomad_conservative_MAG_archaea_scaffold.xlsx", sheet = 1
)
genomad_chromosomes <- read_excel(
  "/path/to/genomad_conservative_MAG_archaea_chromosome.xlsx", sheet = 1
)
genomad_complete_genomes <- read.table(
  "/path/to/genomad_conservative_MAG_archaea_complete_genome.txt",
  stringsAsFactors = FALSE, header = TRUE
)
genomad_PRJNA1162170 <- read_excel(
  "/path/to/genomad_conservative_PRJNA1162170.xlsx", sheet = 1
)
genomad_Zhang_public_Asgard_add <- read_excel(
  "/path/to/genomad_conservative_Zhang_public_Asgard_add.xlsx", sheet = 1
)

genomad <- rbind(
  genomad_contigs,
  genomad_scaffolds,
  genomad_chromosomes,
  genomad_complete_genomes,
  genomad_PRJNA1162170,
  genomad_Zhang_public_Asgard_add
)

##############################
# 2. Load CAT-flagged contigs
##############################
cat_Asgardarchaeota <- read.table(
  "/path/to/cat_Asgardarchaeota.txt",
  stringsAsFactors = FALSE, sep = "\t", header = TRUE
)
cat_PRJNA1162170 <- read.table(
  "/path/to/cat_PRJNA1162170.txt",
  stringsAsFactors = FALSE, sep = "\t", header = TRUE
)
cat_Zhang_public_Asgard_add <- read.table(
  "/path/to/cat_Zhang_public_Asgard_add.txt",
  stringsAsFactors = FALSE, sep = "\t", header = TRUE
)

cat_list <- rbind(
  cat_Asgardarchaeota,
  cat_PRJNA1162170,
  cat_Zhang_public_Asgard_add
)

##############################
# 3. Genome / protein inventories
##############################
# Prodigal protein FASTA paths (one .faa per genome)
prodigal <- read.table(
  "/path/to/dereplicated_Asgardarchaeota_add_PRJNA1162170_public_prodigal.txt",
  stringsAsFactors = FALSE, sep = "\t"
)
prodigal_name <- sapply(prodigal$V1, function(u) {
  bn <- strsplit(u, "/")[[1]]
  sub("\\.faa$", "", bn[length(bn)])
})

# eggNOG families retained after paralogue screening (decision table)
eggnog_remove_paralogs <- read_excel(
  "/path/to/eggnog.xlsx", sheet = 21
)

# eggNOG-mapper annotation files present in the working directory
annotations_file <- dir(pattern = "\\.emapper\\.annotations$")

# Genomes excluded from this run (isolates / special cases)
exclude_annotations <- c(
  "GCF_008000775.2_ASM800077v2_genomic.emapper.annotations",
  "GCA_025839675.1_ASM2583967v1_genomic.emapper.annotations",
  "Heimdallarchaeota_archaeon_HC1.emapper.annotations",
  "Heimdallarchaeota_archaeon_SC1.emapper.annotations"
)

target_annotations <- setdiff(
  intersect(annotations_file, paste0(prodigal_name, ".emapper.annotations")),
  exclude_annotations
)

# Output directory for cleaned per-family protein FASTAs
out_dir <- "/path/to/emapper_faa/eggNOG_0.8_remove_paralogs/screen_cultured_MAGs/MAGs_add_PRJNA1162170_public_remove_viruses_and_aberrant_contigs/Asgard_dRep/"
dir.create(out_dir, showWarnings = FALSE, recursive = TRUE)

##############################
# 4. Per-genome decontamination + marker extraction
##############################
# Helper: map protein ID → contig ID
# eggNOG-mapper query IDs are typically <contig>_<orf_index>
protein_to_contig <- function(pid) {
  parts <- strsplit(pid, "_")[[1]]
  paste(parts[1:(length(parts) - 1)], collapse = "_")
}

for (ann_file in target_annotations) {
  genome_id <- sub("\\.emapper\\.annotations$", "", ann_file)
  message("Processing genome: ", genome_id)

  annotations <- read.delim(
    ann_file,
    stringsAsFactors = FALSE,
    comment.char = "#",
    header = FALSE
  )

  # --- Contigs to remove: free virus (geNomad) ∪ CAT exogenous -------------
  genomad_sub <- subset(genomad, genome == genome_id)
  cat_sub     <- subset(cat_list, genome == genome_id)

  # Exclude integrated proviruses; retain only free-virus calls
  viral_contigs <- subset(genomad_sub, topology != "Provirus")$seq_name
  cat_contigs   <- cat_sub[["X..contig"]]
  if (is.null(cat_contigs)) {
    # fallback if column name differs
    cat_contigs <- cat_sub$contig
  }
  vc_contigs <- union(viral_contigs, cat_contigs)

  # Drop proteins encoded on flagged contigs
  protein_contigs <- sapply(annotations$V1, protein_to_contig)
  annotations <- annotations[!(protein_contigs %in% vc_contigs), ]

  # Load protein sequences for this genome
  faa_path <- prodigal$V1[which(prodigal_name == genome_id)]
  if (length(faa_path) != 1) {
    warning("Protein FASTA not found or not unique for ", genome_id)
    next
  }
  protein <- readAAStringSet(faa_path)
  protein_name <- sapply(names(protein), function(v) strsplit(v, " ")[[1]][1])

  # --- For each eggNOG family, keep one representative sequence ------------
  for (y in seq_len(nrow(eggnog_remove_paralogs))) {
    root_id <- as.character(eggnog_remove_paralogs$root[y])

    # Proteins annotated with this eggNOG root family
    hit <- annotations[
      sapply(annotations$V5, function(z) root_id %in% strsplit(z, ",")[[1]]),
    ]
    if (nrow(hit) == 0) next

    # Prefer annotations with a single root-level assignment
    hit_single <- hit[
      sapply(hit$V5, function(z) {
        ranks <- sapply(strsplit(z, ",")[[1]], function(u) strsplit(u, "@")[[1]][2])
        sum(ranks == "1|root") == 1
      }),
    ]
    if (nrow(hit_single) == 0) next

    # Restrict to Archaea / Eukaryota secondary ranks listed for this family
    allowed_second <- strsplit(as.character(eggnog_remove_paralogs[y, "All"]), ";")[[1]]
    allowed_second <- allowed_second[
      sapply(allowed_second, function(t) strsplit(t, "\\|")[[1]][2]) %in%
        c("Archaea", "Eukaryota")
    ]

    hit_second <- hit_single[
      sapply(hit_single$V5, function(w) strsplit(w, ",")[[1]][2]) %in% allowed_second,
    ]
    if (nrow(hit_second) == 0) next

    # Best hit by score (column V3); write one sequence per genome per family
    best_id <- hit_second$V1[order(hit_second$V3)[1]]
    faa_one <- protein[match(best_id, protein_name)]
    names(faa_one) <- genome_id

    family_tag <- strsplit(root_id, "@")[[1]][1]
    out_faa <- file.path(out_dir, paste0(family_tag, ".faa"))
    writeXStringSet(faa_one, out_faa, format = "fasta", append = TRUE)
  }
}

message("Finished decontamination-aware marker extraction.")
################################################################################
```


## A2 — Genome-level contamination frequencies (Fig. 1a)

### Purpose
Convert contig flags into **% of genomes** with ≥1 bacterial, eukaryotic, viral or unclassified contaminant contig. Compare isolates vs MAGs across Asgard, TACK, Euryarchaeota and DPANN.

### Result (manuscript)
Asgard MAGs show the highest frequencies (bacterial **63.33%**; viral **25.59%**). Isolates are nearly clean. The ≥1% burden threshold is a **summary metric only**, not an automatic removal rule (**Extended Data Fig. 2**).

### Data
`data/decontamination/assessment/` (e.g. contamination-by-lineage tables). Summary percentages are embedded in the figure script for layout only and must match the deposited tables.


### Full script: `scripts/figure_reproduction/fig1a_contamination_frequency.py`

```python
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Fig. 1a — Contamination frequency across archaeal groups

Manuscript
----------
Contamination and taxon sampling explain conflicting eukaryote placements

Reproduces the published panel showing genome-level contamination frequencies
(%) for bacterial, eukaryotic, viral and unclassified contaminants in isolate
genomes versus MAG-derived genomes across Asgard, TACK, Euryarchaeota and DPANN.

Methods context
---------------
Contamination patterns were quantified across major archaeal groups. Genome-
level contamination frequency was calculated as the proportion of genomes
containing at least one contaminant contig in a given category. Contaminant
origins were assigned using CAT (bacterial, eukaryotic, unclassified) and
geNomad (viral). Isolate genomes and MAG-derived genomes are shown separately
because contamination burden is concentrated in MAGs, and among MAGs is highest
in Asgard lineages repeatedly proposed as close relatives of eukaryotes.

Data source (deposited with the manuscript)
-------------------------------------------
    data/decontamination/assessment/contamination_by_lineage.xlsx
Summary percentages below match the deposited assessment tables and are
embedded here for figure layout only.

Dependencies
------------
    pip install matplotlib numpy

Outputs (current working directory)
-----------------------------------
    contamination_frequency_archaea.png
    contamination_frequency_archaea.svg
    contamination_frequency_archaea.pdf

Usage
-----
    python scripts/figure_reproduction/fig1a_contamination_frequency.py
"""

import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, Circle, Rectangle, Polygon
from matplotlib.colors import LinearSegmentedColormap

mpl.rcParams["font.family"] = "DejaVu Sans"
mpl.rcParams["svg.fonttype"] = "none"
mpl.rcParams["pdf.fonttype"] = 42

# ---------------------------------------------------------------------------
# Summary statistics
# From data/decontamination/assessment/contamination_by_lineage.xlsx
# Values = % of genomes with ≥1 contaminant contig in each category.
# N = genome counts used as denominators (must match assessment tables).
# ---------------------------------------------------------------------------
groups = ["Asgard", "TACK", "Euryarchaeota", "DPANN"]

# Isolate genomes and MAG-derived genomes per group
isolate_n = [4, 155, 466, 4]
# MAG sample sizes in the deposited assessment (update if tables change)
mag_n = [469, 469, 469, 469]

categories = [
    "Bacterial\ncontamination",
    "Eukaryotic\ncontamination",
    "Viral\ncontamination",
    "Unclassified\ncontamination",
]

# Rows = groups (Asgard, TACK, Euryarchaeota, DPANN)
# Columns = bacterial, eukaryotic, viral, unclassified (%)
isolate_values = np.array([
    [0.00, 0.00, 0.00, 0.00],   # Asgard isolates
    [0.00, 0.00, 0.00, 0.00],   # TACK isolates
    [0.21, 0.00, 1.29, 0.43],   # Euryarchaeota isolates
    [0.00, 0.00, 0.00, 0.00],   # DPANN isolates
])

# Results text: Asgard bacterial 63.33%; viral 25.59% (highest among groups)
mag_values = np.array([
    [63.33, 2.99, 25.59, 17.27],  # Asgard MAGs
    [30.49, 1.07, 12.37, 5.33],   # TACK MAGs
    [45.63, 0.43, 14.29, 14.93],  # Euryarchaeota MAGs
    [43.71, 0.64, 12.79, 9.81],   # DPANN MAGs
])

N_ISOLATE_TOTAL = int(np.sum(isolate_n))  # 629
N_MAG_TOTAL = int(np.sum(mag_n))          # 1876

# ---------------------------------------------------------------------------
# Colours
# ---------------------------------------------------------------------------
blue = "#1455B3"
red = "#E91E24"
purple = "#6D2DA8"
gray = "#5E5E5E"
header_colors = [blue, red, purple, gray]

cmaps = [
    LinearSegmentedColormap.from_list("bluegrad", ["#FFFFFF", "#D9E9F7", blue]),
    LinearSegmentedColormap.from_list("redgrad", ["#FFFFFF", "#F8D6D6", red]),
    LinearSegmentedColormap.from_list("purplegrad", ["#FFFFFF", "#E4D8F1", purple]),
    LinearSegmentedColormap.from_list("graygrad", ["#FFFFFF", "#D9D9D9", gray]),
]

mag_max = mag_values.max(axis=0)

# ---------------------------------------------------------------------------
# Figure
# ---------------------------------------------------------------------------
fig = plt.figure(figsize=(8.8, 10.8), dpi=300)
ax = fig.add_axes([0, 0, 1, 1])
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis("off")

ax.text(0.03, 0.965, "a", fontsize=20, fontweight="bold", va="center")
ax.text(
    0.11, 0.965,
    "Contamination frequency across archaeal groups",
    fontsize=20, fontweight="bold", va="center",
)

col_x = [0.43, 0.58, 0.73, 0.88]
for x, label, color in zip(col_x, categories, header_colors):
    ax.text(
        x, 0.905, label, ha="center", va="center",
        fontsize=12, fontweight="bold", color=color,
    )

# --- Column icons (schematic) ---
# Bacterial
x, y = col_x[0], 0.82
for dx, dy, angle in [(-0.018, 0.010, 25), (0.022, -0.020, 25)]:
    box = FancyBboxPatch(
        (x + dx - 0.020, y + dy - 0.038), 0.040, 0.076,
        boxstyle="round,pad=0.005,rounding_size=0.02",
        facecolor="#BDD5F3", edgecolor=blue, linewidth=1.2,
    )
    t = mpl.transforms.Affine2D().rotate_deg_around(x + dx, y + dy, angle) + ax.transData
    box.set_transform(t)
    ax.add_patch(box)

# Eukaryotic
x, y = col_x[1], 0.82
ax.add_patch(Circle((x, y), 0.054, facecolor="#FAD1D1", edgecolor=red, linewidth=1.2))
ax.add_patch(Circle((x, y), 0.022, facecolor="#F38A8D", edgecolor=red, linewidth=1.0))

# Viral
x, y = col_x[2], 0.82
ax.add_patch(Circle((x, y), 0.048, facecolor="#D9C2EE", edgecolor=purple, linewidth=1.2))
for ang in np.linspace(0, 2 * np.pi, 8, endpoint=False):
    x1, y1 = x + 0.050 * np.cos(ang), y + 0.050 * np.sin(ang)
    x2, y2 = x + 0.062 * np.cos(ang), y + 0.062 * np.sin(ang)
    ax.plot([x1, x2], [y1, y2], color=purple, lw=1.1)
    ax.add_patch(Circle((x2, y2), 0.0045, facecolor="#D9C2EE", edgecolor=purple, linewidth=1.0))
theta = np.linspace(0, 2 * np.pi, 7)[:-1] + np.pi / 6
pts = np.c_[x + 0.026 * np.cos(theta), y + 0.026 * np.sin(theta)]
ax.add_patch(Polygon(pts, closed=True, fill=False, edgecolor=purple, linewidth=1.0))

# Unclassified
x, y = col_x[3], 0.82
ax.add_patch(Circle((x, y), 0.054, facecolor="#F1F1F1", edgecolor=gray, linewidth=1.2))
ax.text(x, y - 0.003, "?", ha="center", va="center", fontsize=42, fontweight="bold", color=gray)

ax.text(0.57, 0.72, "Contamination frequency (%)", fontsize=18, fontweight="bold", ha="center")

group_label_x0 = 0.19
table_x0 = 0.37
table_x1 = 0.96
row_h = 0.064
col_edges = np.linspace(table_x0, table_x1, 5)


def rounded_outer_box(x0, y0, w, h, edgecolor="black", facecolor="white", lw=1.6):
    patch = FancyBboxPatch(
        (x0, y0), w, h,
        boxstyle="round,pad=0.002,rounding_size=0.028",
        facecolor=facecolor, edgecolor=edgecolor, linewidth=lw,
    )
    ax.add_patch(patch)
    return patch


# Side labels: isolate vs MAG
iso_y0 = 0.435
iso_h = row_h * 4
rounded_outer_box(0.035, iso_y0, 0.145, iso_h, edgecolor="black", facecolor="#EFF5FC", lw=1.5)
ax.text(
    0.108, iso_y0 + iso_h * 0.62, "Isolate\ngenomes",
    ha="center", va="center", fontsize=13, fontweight="bold", color=blue,
)
ax.text(
    0.108, iso_y0 + 0.035, f"(N={N_ISOLATE_TOTAL})",
    ha="center", va="center", fontsize=12, fontweight="bold", color=blue,
)

mag_y0 = 0.155
mag_h = row_h * 4
rounded_outer_box(0.035, mag_y0, 0.145, mag_h, edgecolor="black", facecolor="#F4ECFA", lw=1.5)
ax.text(
    0.108, mag_y0 + mag_h * 0.73, "MAG-derived\ngenomes",
    ha="center", va="center", fontsize=13, fontweight="bold", color=purple,
)
ax.text(
    0.108, mag_y0 + 0.040, f"(N={N_MAG_TOTAL})",
    ha="center", va="center", fontsize=12, fontweight="bold", color=purple,
)


def draw_section(y0, values, ns, use_heatmap):
    """Draw one block (isolates or MAGs): group labels + frequency grid."""
    section_h = row_h * 4
    rounded_outer_box(
        group_label_x0, y0, table_x1 - group_label_x0, section_h,
        edgecolor="black", facecolor="white", lw=1.6,
    )
    for i in range(1, 4):
        y = y0 + i * row_h
        ax.plot([group_label_x0, table_x1], [y, y], color="#C9C9C9", lw=0.8)
    ax.plot([table_x0, table_x0], [y0, y0 + section_h], color="#C9C9C9", lw=0.8)
    for xe in col_edges[1:-1]:
        ax.plot([xe, xe], [y0, y0 + section_h], color="#C9C9C9", lw=0.8)

    for r, (group, n) in enumerate(zip(groups, ns)):
        yy = y0 + section_h - (r + 0.5) * row_h
        ax.text(
            group_label_x0 + 0.012, yy, f"{group}\n(N = {n})",
            ha="left", va="center", fontsize=11.5, fontweight="bold",
        )
        for c in range(4):
            xleft, xright = col_edges[c], col_edges[c + 1]
            val = values[r, c]
            if use_heatmap:
                norm_val = val / mag_max[c] if mag_max[c] > 0 else 0
                ax.add_patch(
                    Rectangle(
                        (xleft, yy - row_h / 2), xright - xleft, row_h,
                        facecolor=cmaps[c](norm_val), edgecolor="none",
                    )
                )
            if use_heatmap and val >= 0.70 * mag_max[c]:
                text_color = "white"
            elif not use_heatmap and val == 0:
                text_color = header_colors[c]
            else:
                text_color = "black"
            ax.text(
                (xleft + xright) / 2, yy, f"{val:.2f}",
                ha="center", va="center", fontsize=12,
                fontweight="bold", color=text_color,
            )


draw_section(iso_y0, isolate_values, isolate_n, use_heatmap=False)
draw_section(mag_y0, mag_values, mag_n, use_heatmap=True)

ax.text(0.57, 0.112, "Contamination frequency (%)", fontsize=17, fontweight="bold", ha="center")

# Colour bars (scale = max frequency per category among MAGs)
bar_y = 0.079
bar_h = 0.012
for c in range(4):
    x0, x1 = col_edges[c], col_edges[c + 1]
    grad = np.linspace(0, 1, 256).reshape(1, -1)
    ax.imshow(
        grad, extent=[x0 + 0.010, x1 - 0.010, bar_y, bar_y + bar_h],
        origin="lower", aspect="auto", cmap=cmaps[c], interpolation="bicubic",
    )
    ax.text(x0 + 0.003, bar_y - 0.014, "0", ha="left", va="center", fontsize=10.5)
    ax.text(x1 - 0.003, bar_y - 0.014, f"{mag_max[c]:.2f}", ha="right", va="center", fontsize=10.5)

ax.text(
    0.57, 0.030,
    "Color intensity based on frequency within each contamination category.",
    fontsize=12, fontweight="bold", ha="center",
)

plt.savefig("contamination_frequency_archaea.png", dpi=600, bbox_inches="tight", facecolor="white")
plt.savefig("contamination_frequency_archaea.svg", bbox_inches="tight", facecolor="white")
plt.savefig("contamination_frequency_archaea.pdf", bbox_inches="tight", facecolor="white")
print("Wrote contamination_frequency_archaea.{png,svg,pdf}")
```


## A3 — Asgard lineages and contaminant eukaryote-like proteins (Fig. 1b–c)

### Purpose
Test whether contamination is further concentrated in candidate closest-relative Asgard lineages, and whether proteins on contaminant contigs can look eukaryote-like—providing a mechanism for misleading affinity.

### Result (manuscript)
Hodarchaeales, Njordarchaeales and related groups rank among the most contaminated. Contaminant-derived proteins with apparent eukaryotic similarity are enriched in the same groups (**Fig. 1b–c**, Extended Data Figs. 5–6).

Fig. 1b is assembled in Prism from lineage-level tables under `data/decontamination/assessment/`. Fig. 1c is scripted below.


### Supporting analysis for Fig. 1c — eukaryote-assigned proteins on exogenous contigs

**Fig. 1c** reports the percentage of MAGs that harbour contaminant-derived proteins with apparent similarity to eukaryotic homologues. Operationally, ORF-level CAT classifications (`*.ORF2LCA.txt`) are used to count proteins assigned to **Eukaryota** (`1;131567;2759`) that sit on contigs already flagged as candidate exogenous sequences (bacterial, eukaryotic, viral or chimeric).

1. **`run_CAT_for_eukaryote_assigned_proteins.sh`** — run CAT contigs mode per genome to produce ORF2LCA tables.  
2. **`count_eukaryote_like_proteins_from_CAT.R`** — count Eukaryota-classified ORFs on flagged contigs and summarise per genome / category.  
3. Summary frequencies are deposited under `data/decontamination/assessment/` and plotted by `fig1c_contamination_derived_eukaryote_like_proteins.py`.


### Full script: `scripts/decontamination/run_CAT_for_eukaryote_assigned_proteins.sh`

Run CAT in **contigs** mode so that each genome produces an `*.ORF2LCA.txt` table. These ORF-level lineages are used downstream to count proteins classified as Eukaryota on contigs already flagged as candidate exogenous sequences (bacterial, eukaryotic, viral or chimeric).

```bash
#!/bin/bash
# ============================================================
# Script : run_CAT_for_eukaryote_assigned_proteins.sh (EXAMPLE TEMPLATE)
# Purpose: Run CAT in contigs mode to generate ORF2LCA.txt files
#          used for identifying eukaryote-assigned proteins on
#          candidate exogenous contigs (Fig. 1c support).
#
# Manuscript
# ----------
# Contamination and taxon sampling explain conflicting eukaryote placements
#
# Methods context
# ---------------
# Contaminant-derived proteins with apparent similarity to eukaryotic
# homologues were identified by inspecting ORF-level CAT classifications
# (ORF2LCA) on contigs previously flagged as exogenous. CAT contigs mode
# must be run with the same database/taxonomy used for contig flagging.
#
# Tools  : CAT_pack (CAT v6.0), sensitive mode
# Input  : one genome FASTA (.fna / .fa / .fasta)
# Output : ${OUTPUT_DIR}/cat_classification/contigs/${base}.ORF2LCA.txt
#          (and related CAT contig outputs)
# Note   : EXAMPLE template only. Paths are placeholders.
# ============================================================

set -euo pipefail

# ----- User-configurable parameters -----
genome_fna="$1"                  # input genome fasta
base=$(basename "${genome_fna}" .fna)
base=${base%.fa}
base=${base%.fasta}

CAT_DB="/path/to/CAT_database"   # CAT database directory
CAT_TAX="/path/to/CAT_taxonomy"  # CAT taxonomy directory
OUTPUT_DIR="/path/to/output"
THREADS_CAT=16

mkdir -p "${OUTPUT_DIR}/cat_classification/contigs"
mkdir -p "${OUTPUT_DIR}/logs"

# ----- Run CAT (contigs mode; writes ORF2LCA among other outputs) -----
CAT_pack contigs \
    -c "${genome_fna}" \
    -d "${CAT_DB}" \
    -t "${CAT_TAX}" \
    -o "${OUTPUT_DIR}/cat_classification/contigs/${base}" \
    --sensitive \
    -n "${THREADS_CAT}" \
    > "${OUTPUT_DIR}/logs/${base}.CAT_contigs.log" 2>&1

echo "Finished CAT classification for ${base}"
echo "Expected ORF table: ${OUTPUT_DIR}/cat_classification/contigs/${base}.ORF2LCA.txt"
```


### Full script: `scripts/decontamination/count_eukaryote_like_proteins_from_CAT.R`

Count proteins whose CAT ORF lineage is **Eukaryota** (`1;131567;2759`) and that lie on contigs already annotated as bacterial, eukaryotic, viral or chimeric exogenous candidates. Per-genome counts feed the assessment tables underlying **Fig. 1c** (and Extended Data lineage summaries). Objects such as `cat_bacteria`, `cat_eukaryota`, `cat_viruses` and `cat_chimeric` are assumed to be loaded from the deposited flag tables (or equivalent local parses of CAT / geNomad outputs).

```r
#!/usr/bin/env Rscript
################################################################################
# Script : count_eukaryote_like_proteins_from_CAT.R (EXAMPLE TEMPLATE)
# Purpose: Count proteins classified as Eukaryota (NCBI lineage prefix
#          1;131567;2759) on contigs previously flagged as bacterial,
#          eukaryotic, viral or chimeric exogenous sequences.
#
# Manuscript
# ----------
# Contamination and taxon sampling explain conflicting eukaryote placements
#
# Methods / Results context
# -------------------------
# Proteins encoded on contaminant-derived contigs were examined for apparent
# affinity to eukaryotic homologues. ORF-level CAT classifications (ORF2LCA)
# provide the taxonomic assignment of each predicted protein. Counts are
# summarised per genome and per contaminant category and deposited under
# data/decontamination/assessment/ for Fig. 1c.
#
# Prerequisites
# -------------
# - CAT contigs mode has been run (see run_CAT_for_eukaryote_assigned_proteins.sh)
# - Flag tables for exogenous contigs are available (e.g. cat_bacteria,
#   cat_eukaryota, cat_viruses / geNomad free-virus, cat_chimeric), each with
#   columns identifying genome and contig (e.g. genome, X..contig)
# - ORF2LCA files are discoverable by genome ID under a local search path
#
# Note: EXAMPLE template. Replace ORF2LCA search paths and input tables
# with local paths. Do not hard-code production cluster directories in deposits.
################################################################################

# ---------------------------
# 1. Helper: safely read ORF2LCA file
# ---------------------------
# Edit ORF2LCA_DIRS to the local directories that store *.ORF2LCA.txt
ORF2LCA_DIRS <- c(
  "/path/to/CAT_pack/contigs/Asgardarchaeota",
  "/path/to/CAT_pack/contigs/Thermoproteota",
  "/path/to/CAT_pack/contigs/other_batches"
)

read_ORF2LCA <- function(genome_id, search_dirs = ORF2LCA_DIRS) {
  for (dir in search_dirs) {
    path <- file.path(dir, paste0(genome_id, ".ORF2LCA.txt"))
    if (file.exists(path)) {
      return(read.delim(path, stringsAsFactors = FALSE, sep = "\t", header = TRUE))
    }
  }
  warning(paste("ORF2LCA file not found for genome:", genome_id))
  return(NULL)
}

# ---------------------------
# 2. Core: count Eukaryota proteins on target contigs
# ---------------------------
count_eukaryotic_proteins <- function(genome_id, target_contigs) {
  ORF2LCA <- read_ORF2LCA(genome_id)
  if (is.null(ORF2LCA) || nrow(ORF2LCA) == 0) {
    return(0)
  }

  # Contig ID from ORF name (drop the trailing _ORF token)
  orf_col <- if ("X..ORF" %in% names(ORF2LCA)) "X..ORF" else names(ORF2LCA)[1]
  ORF2LCA$contig <- sapply(ORF2LCA[[orf_col]], function(y) {
    parts <- strsplit(as.character(y), "_", fixed = TRUE)[[1]]
    if (length(parts) <= 1) {
      return(as.character(y))
    }
    paste(parts[1:(length(parts) - 1)], collapse = "_")
  })

  idx <- which(ORF2LCA$contig %in% target_contigs)
  if (length(idx) == 0) {
    return(0)
  }

  # First three ranks of the CAT lineage string
  lineages <- sapply(ORF2LCA$lineage[idx], function(z) {
    paste(strsplit(as.character(z), ";", fixed = TRUE)[[1]][1:3], collapse = ";")
  })

  # Eukaryota under cellular organisms: 1;131567;2759
  sum(lineages == "1;131567;2759", na.rm = TRUE)
}

# ---------------------------
# 3. Per-category counts
#    Expect objects already in the R session (or load from deposited TSVs):
#      cat_bacteria, cat_eukaryota, cat_viruses, cat_chimeric
#    with columns: genome, X..contig (or rename below)
# ---------------------------

# --- 3.1 Bacterial contigs ---
cat_bacteria_eukaryotic_protein <- sapply(unique(cat_bacteria$genome), function(x) {
  contigs <- subset(cat_bacteria, genome == x)$X..contig
  count_eukaryotic_proteins(x, contigs)
})

# --- 3.2 Eukaryotic contigs ---
cat_eukaryota_eukaryotic_protein <- sapply(unique(cat_eukaryota$genome), function(x) {
  contigs <- subset(cat_eukaryota, genome == x)$X..contig
  count_eukaryotic_proteins(x, contigs)
})

# --- 3.3 Viral contigs (CAT viral flags and/or geNomad free-virus) ---
# Adapt genome/contig columns to the merged viral flag table used locally.
viral_genomes <- unique(cat_viruses$genome)
cat_viruses_eukaryotic_protein <- sapply(viral_genomes, function(x) {
  contigs <- subset(cat_viruses, genome == x)$X..contig
  count_eukaryotic_proteins(x, contigs)
})

# --- 3.4 Chimeric / unclassified exogenous contigs ---
cat_chimeric_eukaryotic_protein <- sapply(unique(cat_chimeric$genome), function(x) {
  contigs <- subset(cat_chimeric, genome == x)$X..contig
  count_eukaryotic_proteins(x, contigs)
})

# ---------------------------
# 4. Optional: write per-genome summary
# ---------------------------
# results <- data.frame(
#   genome = names(cat_bacteria_eukaryotic_protein),
#   bacteria_euk_proteins = as.integer(cat_bacteria_eukaryotic_protein),
#   stringsAsFactors = FALSE
# )
# write.table(results, file = "eukaryote_like_protein_counts.tsv",
#             sep = "\t", quote = FALSE, row.names = FALSE)

message("Analysis completed.")
```


### Full script: `scripts/figure_reproduction/fig1c_contamination_derived_eukaryote_like_proteins.py`

```python
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Fig. 1c — MAGs harbouring contamination-derived eukaryote-like proteins (%)

Manuscript
----------
Contamination and taxon sampling explain conflicting eukaryote placements

Reproduces the published panel: percentage of MAGs in each major archaeal
group that encode proteins located on contaminant contigs and showing
apparent similarity to eukaryotic homologues (bacterial-, eukaryotic-,
viral- and unclassified-derived categories).

Methods / Results context
-------------------------
Proteins encoded on contaminant-derived contigs were analysed for apparent
affinity to eukaryotic homologues. Such proteins were enriched in Asgard
genomes and were especially frequent in lineages with elevated contamination
burdens. Many of the strongest apparent eukaryote-like matches originated
from genomic regions classified as contaminants, providing a direct mechanism
by which contamination can increase apparent similarity between specific
Asgard lineages and eukaryotes and thereby generate misleading
archaeal–eukaryotic affinities.

Data source (deposited with the manuscript)
-------------------------------------------
    data/decontamination/assessment/contamination_derived_eukaryote_like_protein_frequency.xlsx
Summary percentages below match the deposited assessment tables and are
embedded here for figure layout only.

Dependencies
------------
    pip install matplotlib numpy

Outputs (current working directory)
-----------------------------------
    contamination_derived_eukaryote_like_proteins.png
    contamination_derived_eukaryote_like_proteins.svg
    contamination_derived_eukaryote_like_proteins.pdf

Usage
-----
    python scripts/figure_reproduction/fig1c_contamination_derived_eukaryote_like_proteins.py
"""

import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, FancyBboxPatch, Polygon
from matplotlib.gridspec import GridSpec

mpl.rcParams["font.family"] = "DejaVu Sans"
mpl.rcParams["svg.fonttype"] = "none"
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["axes.linewidth"] = 1.2

# ---------------------------------------------------------------------------
# Summary statistics
# From contamination_derived_eukaryote_like_protein_frequency.xlsx
# values = % of MAGs with ≥1 contamination-derived protein showing
#          apparent similarity to eukaryotic homologues
# Order: Asgard, TACK, Euryarchaeota, DPANN
# ---------------------------------------------------------------------------
groups = ["Asgard", "TACK", "Euryarchaeota", "DPANN"]
n = 469  # MAG count per group (denominator); confirm against assessment tables

datasets = [
    {
        "title": "Bacterial-derived",
        "color": "#1F5FAF",
        "values": [0.85, 0.21, 0.00, 0.21],
        "xmax": 1.00,
        "xticks": [0.00, 0.25, 0.50, 0.75, 1.00],
    },
    {
        "title": "Eukaryotic-derived",
        "color": "#ED1C24",
        "values": [2.99, 1.07, 0.43, 0.64],
        "xmax": 3.00,
        "xticks": [0, 1, 2, 3],
    },
    {
        "title": "Viral-derived",
        "color": "#6F2DA8",
        "values": [1.71, 0.85, 0.21, 0.64],
        "xmax": 2.00,
        "xticks": [0.0, 0.5, 1.0, 1.5, 2.0],
    },
    {
        "title": "Unclassified-derived",
        "color": "#666666",
        "values": [0.64, 0.00, 0.00, 0.00],
        "xmax": 0.80,
        "xticks": [0.0, 0.2, 0.4, 0.6, 0.8],
    },
]

# ---------------------------------------------------------------------------
# Figure
# ---------------------------------------------------------------------------
fig = plt.figure(figsize=(18, 4.8), dpi=300)
gs = GridSpec(
    1, 5,
    width_ratios=[1.05, 2.15, 2.15, 2.15, 2.15],
    wspace=0.32, left=0.025, right=0.995, top=0.78, bottom=0.17,
)

fig.text(0.015, 0.92, "c", fontsize=19, fontweight="bold", va="center")
fig.text(
    0.04, 0.92,
    "MAGs harbouring contamination-derived eukaryote-like proteins (%)",
    fontsize=20, fontweight="bold", va="center",
)

ax_labels = fig.add_subplot(gs[0, 0])
ax_labels.set_xlim(0, 1)
ax_labels.set_ylim(-0.5, 3.5)
ax_labels.axis("off")
ypos = np.arange(len(groups))[::-1]
for y, g in zip(ypos, groups):
    ax_labels.text(
        0.02, y, f"{g}\n(N={n})",
        ha="left", va="center", fontsize=12.5, fontweight="bold",
    )


def draw_bacteria_icon(ax, x, y, color):
    for dx, dy, ang in [(-0.05, 0.02, 25), (0.05, -0.03, 25)]:
        patch = FancyBboxPatch(
            (x + dx - 0.045, y + dy - 0.09), 0.09, 0.18,
            boxstyle="round,pad=0.01,rounding_size=0.05",
            facecolor="#BFD5F2", edgecolor=color, linewidth=1.2,
            transform=ax.transAxes, clip_on=False,
        )
        t = mpl.transforms.Affine2D().rotate_deg_around(x + dx, y + dy, ang) + ax.transAxes
        patch.set_transform(t)
        ax.add_patch(patch)


def draw_euk_icon(ax, x, y, color):
    ax.add_patch(Circle(
        (x, y), 0.09, transform=ax.transAxes,
        facecolor="#F7C7C7", edgecolor=color, linewidth=1.2, clip_on=False,
    ))
    ax.add_patch(Circle(
        (x, y), 0.038, transform=ax.transAxes,
        facecolor="#F27D7D", edgecolor=color, linewidth=1.0, clip_on=False,
    ))


def draw_virus_icon(ax, x, y, color):
    ax.add_patch(Circle(
        (x, y), 0.075, transform=ax.transAxes,
        facecolor="#D8C1EC", edgecolor=color, linewidth=1.2, clip_on=False,
    ))
    for ang in np.linspace(0, 2 * np.pi, 8, endpoint=False):
        x1, y1 = x + 0.077 * np.cos(ang), y + 0.077 * np.sin(ang)
        x2, y2 = x + 0.103 * np.cos(ang), y + 0.103 * np.sin(ang)
        ax.plot([x1, x2], [y1, y2], color=color, lw=1.1, transform=ax.transAxes, clip_on=False)
        ax.add_patch(Circle(
            (x2, y2), 0.008, transform=ax.transAxes,
            facecolor="#D8C1EC", edgecolor=color, linewidth=1.0, clip_on=False,
        ))
    theta = np.linspace(0, 2 * np.pi, 7)[:-1] + np.pi / 6
    pts = np.c_[x + 0.04 * np.cos(theta), y + 0.04 * np.sin(theta)]
    ax.add_patch(Polygon(
        pts, closed=True, fill=False, edgecolor=color, linewidth=1.0,
        transform=ax.transAxes, clip_on=False,
    ))


def draw_unknown_icon(ax, x, y, color):
    ax.add_patch(Circle(
        (x, y), 0.09, transform=ax.transAxes,
        facecolor="#EEEEEE", edgecolor=color, linewidth=1.2, clip_on=False,
    ))
    ax.text(
        x, y - 0.004, "?", transform=ax.transAxes,
        ha="center", va="center", fontsize=34, fontweight="bold", color=color,
    )


icon_drawers = [draw_bacteria_icon, draw_euk_icon, draw_virus_icon, draw_unknown_icon]
icon_x = [0.83, 0.84, 0.84, 0.80]

for i, item in enumerate(datasets):
    ax = fig.add_subplot(gs[0, i + 1])
    vals = np.array(item["values"])
    color = item["color"]
    xmax = item["xmax"]

    ax.set_xlim(0, xmax)
    ax.set_ylim(-0.25, 3.45)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_visible(False)
    ax.spines["bottom"].set_linewidth(1.2)
    ax.tick_params(axis="y", left=False, labelleft=False)
    ax.tick_params(axis="x", width=1.0, length=6, labelsize=10.5)
    ax.set_xticks(item["xticks"])

    if i == 0:
        ax.set_xticklabels([f"{x:.2f}" for x in item["xticks"]])
    elif i in (2, 3):
        ax.set_xticklabels([f"{x:.1f}" for x in item["xticks"]])
    else:
        ax.set_xticklabels([str(int(x)) for x in item["xticks"]])

    for y in ypos:
        ax.hlines(y, 0, xmax, color="#8D8D8D", lw=1.6, linestyles="dotted", zorder=1)

    for y, val in zip(ypos, vals):
        ax.hlines(y, 0, val, color=color, lw=2.1, zorder=2)
        ax.scatter(val, y, s=135, color=color, edgecolor=color, zorder=3)
        xtext = min((xmax * 0.06 if val == 0 else val + xmax * 0.05), xmax * 0.96)
        ax.text(
            xtext, y, f"{val:.2f}",
            color=color, fontsize=11.5, fontweight="bold",
            ha="left", va="center",
            bbox=dict(facecolor="white", edgecolor="none", pad=0.3),
        )

    ax.text(
        0.03, 1.18, item["title"],
        transform=ax.transAxes, color=color, fontsize=17, fontweight="bold",
        ha="left", va="center", clip_on=False,
    )
    icon_drawers[i](ax, icon_x[i], 1.18, color)

    ax.set_xlabel(
        "Contamination-derived eukaryotic proteins (%)",
        fontsize=10.8, fontweight="bold", labelpad=6,
    )

plt.savefig(
    "contamination_derived_eukaryote_like_proteins.png",
    dpi=600, bbox_inches="tight", facecolor="white",
)
plt.savefig(
    "contamination_derived_eukaryote_like_proteins.svg",
    bbox_inches="tight", facecolor="white",
)
plt.savefig(
    "contamination_derived_eukaryote_like_proteins.pdf",
    bbox_inches="tight", facecolor="white",
)
print("Wrote contamination_derived_eukaryote_like_proteins.{png,svg,pdf}")
```


### Extended Data — geNomad vs Phager overlap

Euler diagrams of viral-call overlap by archaeal group. Input: `data/viral_detection_comparison/summary_venn_counts.xlsx` with columns `group`, `geNomad_only`, `Phager_only`, `shared`.


### Full script: `scripts/figure_reproduction/plot_extended_data_fig1_venn.R`

```r
################################################################################
# Extended Data — Viral detection overlap (geNomad vs Phager)
#
# Manuscript
# ----------
# Contamination and taxon sampling explain conflicting eukaryote placements
#
# Methods context
# ---------------
# Viral sequences were identified primarily with geNomad (conservative mode)
# during primary decontamination. An independent viral-calling tool (Phager)
# was used for comparison of detection overlap across major archaeal MAG
# groups. Free-virus calls (not proviruses) informed contaminant removal;
# method concordance is summarised as three-way counts (geNomad only, Phager
# only, shared) and plotted as Euler diagrams.
#
# Input
# -----
#   data/viral_detection_comparison/summary_venn_counts.xlsx
#   Required columns: group, geNomad_only, Phager_only, shared
#
# Output
# ------
#   data/viral_detection_comparison/venn_<group>.pdf
#
# Dependencies
# ------------
#   install.packages(c("eulerr", "readxl"))
################################################################################

library(eulerr)
library(readxl)

input_xlsx <- "data/viral_detection_comparison/summary_venn_counts.xlsx"
out_dir    <- "data/viral_detection_comparison"

if (!file.exists(input_xlsx)) {
  stop("Missing input: ", input_xlsx)
}
dir.create(out_dir, showWarnings = FALSE, recursive = TRUE)

venn_summary <- as.data.frame(read_excel(input_xlsx, sheet = 1))
required <- c("group", "geNomad_only", "Phager_only", "shared")
missing_cols <- setdiff(required, names(venn_summary))
if (length(missing_cols) > 0) {
  stop("Table missing columns: ", paste(missing_cols, collapse = ", "))
}

for (i in seq_len(nrow(venn_summary))) {
  grp <- as.character(venn_summary$group[i])
  message("Plotting: ", grp)

  fit <- euler(c(
    geNomad = venn_summary$geNomad_only[i],
    Phager  = venn_summary$Phager_only[i],
    "geNomad&Phager" = venn_summary$shared[i]
  ))

  outfile <- file.path(out_dir, paste0("venn_", grp, ".pdf"))
  pdf(outfile, width = 5, height = 5)
  print(plot(
    fit,
    fills = list(fill = c("#1F77B4", "#FF7F0E"), alpha = 0.5),
    edges = list(col = "black", lwd = 2),
    quantities = list(font = 2, cex = 1.3),
    labels = list(font = 2, cex = 1.1),
    main = paste0(grp, " MAGs")
  ))
  dev.off()
  message("  Wrote ", outfile)
}

message("Done. Euler diagrams written under ", out_dir)
################################################################################
```


### End of Part A

Contamination is **not** background: it is enriched in Asgard and in several lineages repeatedly proposed as close relatives of eukaryotes. Part B asks what happens to the tree when this signal—and taxonomic sampling imbalance—are controlled.


---
# Part B — Phylogenomic analysis

**Results blocks:** factorial design → distinct effects of the two biases → full-control topology → independent tests.

**Logic**
1. **B1** Decontaminate each phylogenomic genome set (GS) → raw vs clean.  
2. **B2** Hierarchical taxonomic balancing → GS-*-B (Asgard ≈ TACK; Euryarchaeota/DPANN modestly lower).  
3. **B3** Four independently curated PMSs after decontamination and single-protein-tree screening.  
4. **B4** Factorial ML (all GS × all PMS) under LG+C60+F+G → **Fig. 2**.  
5. **B5** Independent tests: post-hoc audit, expanded sampling, PMSF, AU, CAT-GTR → **Fig. 3**.

**Main result under full control:** all **12** decontaminated × balanced analyses recover eukaryotes as sister to a monophyletic **TACK–Asgard** radiation, outside currently sampled Asgard subgroups.


## B1 — Decontamination of phylogenomic genome collections

### Purpose
Produce **clean** GS collections for concatenation (operational step for Fig. 2), and record **contigs removed (%)**.

### Manuscript numbers (Extended Data Table 2)
| Genome set | Contigs removed (%) | Sampling |
| --- | --- | --- |
| GS-Zhang2025-B-clean | **4.91** | Balanced |
| GS-Liu2021-B-clean | **4.22** | Balanced |
| GS-Present-B-clean | **5.11** | Balanced |

Moderate removal (~4–5%) is argued to eliminate artefactual phylogenetic signal while retaining most sequence and core markers. Use the same CAT + geNomad pipeline as Part A, restricted to each GS accession list (scripts in A1).


## B2 — Hierarchical taxonomic balancing

### Purpose
Build balanced collections so sampling imbalance can be tested **separately** from contamination.

### Target representation (Methods)
Major balancing goal: **Asgard and TACK** at comparable numbers; **Euryarchaeota and DPANN** at modestly lower numbers. Bacterial and eukaryotic isolates are included at numbers roughly equivalent to a single archaeal phylum.

### Algorithm
At each rank: count lineages → sort small→large → `average = remaining_target / remaining_lineages` → keep small lineages in full → drill into large lineages at the next rank → record `n_retain` in `sampling.xlsx` → execution samples/copies FASTA.

The example script below balances **Thermoproteota (TACK)** for **GS-Present-B**. Analogous scripts are used for other phyla. Balancing was defined **before** tree inference and is independent of the resulting eukaryotic placement.


### Full script: `scripts/balancing/hierarchical_balancing_TACK_GS-Present-B.R`

```r
################################################################################
# Hierarchical taxonomic balancing — TACK archaea (Thermoproteota)
# Dataset: GS-Present-B
#
# Manuscript
# ----------
# Contamination and taxon sampling explain conflicting eukaryote placements
#
# Purpose
# -------
# Construct a taxonomically balanced representation of TACK archaea for the
# independently assembled genome collection GS-Present-B. Balancing was
# performed hierarchically so that major archaeal groups (Asgard and TACK)
# were brought to comparable representation, while Euryarchaeota and DPANN
# were retained at modestly lower numbers.
#
# Core algorithm (as implemented and described in Methods)
# --------------------------------------------------------
# 1. At a given taxonomic rank, count the number of genomes in each lineage.
# 2. Sort lineages from smallest to largest.
# 3. average = remaining_target / number_of_lineages_still_to_be_processed.
# 4. If a lineage has ≤ average genomes → retain ALL of them.
# 5. If a lineage has > average genomes → descend to the next lower rank
#    (order → family → genus → species) and repeat the same calculation.
# 6. Final retention numbers for each terminal lineage are recorded in
#    sampling.xlsx.
# 7. The execution stage samples genomes according to sampling.xlsx and
#    copies the selected FASTA files to the output directory.
#
# Input
# -----
# - GTDB release metadata (ar53_metadata_r220.tsv)
# - GTDB-Tk summary tables (contig / scaffold / chromosome / complete)
# - dRep-dereplicated genome directory
# - sampling.xlsx (decision table prepared during the exploration stage)
#
# Output
# ------
# - Selected Thermoproteota genome FASTA files for GS-Present-B
################################################################################

##############################
# Stage 1: Data preparation
##############################

# --- 1.1 High-quality RefSeq isolate complete genomes ----------------------
# Filter criteria:
#   - RS_ accessions (RefSeq)
#   - ncbi_genome_category == "none"
#   - Complete Genome assembly level
#   - CheckM2 completeness ≥ 70%
#   - CheckM2 contamination ≤ 10%

ar53_metadata_r220 <- read.delim(
  "/path/to/ar53_metadata_r220.tsv",
  stringsAsFactors = FALSE,
  header = TRUE
)

ar53_metadata_r220_RS <- ar53_metadata_r220[
  substr(ar53_metadata_r220$accession, 1, 3) == "RS_" &
    ar53_metadata_r220$ncbi_genome_category == "none" &
    ar53_metadata_r220$ncbi_assembly_level == "Complete Genome" &
    ar53_metadata_r220$checkm2_completeness >= 70 &
    ar53_metadata_r220$checkm2_contamination <= 10,
]

# Extract Thermoproteota isolates (TACK component)
phylum <- sapply(ar53_metadata_r220_RS$gtdb_taxonomy, function(x) {
  strsplit(x, ";")[[1]][2]
})
RS_Thermoproteota <- ar53_metadata_r220_RS[phylum == "p__Thermoproteota", ]
RS_Thermoproteota_class <- sapply(RS_Thermoproteota$gtdb_taxonomy, function(x) {
  strsplit(x, ";")[[1]][3]
})

# --- 1.2 GTDB-Tk classifications for MAGs at all assembly levels -----------
gtdbtk.ar53_contig <- read.table(
  "/path/to/gtdbtk/MAG_archaea_contig_new/gtdbtk.ar53.summary.tsv",
  stringsAsFactors = FALSE, sep = "\t", header = TRUE
)
gtdbtk.ar53_scaffold <- read.table(
  "/path/to/gtdbtk/MAG_archaea_scaffold_new/gtdbtk.ar53.summary.tsv",
  stringsAsFactors = FALSE, sep = "\t", header = TRUE
)
gtdbtk.ar53_chromosome <- read.table(
  "/path/to/gtdbtk/MAG_archaea_chromosome_new/gtdbtk.ar53.summary.tsv",
  stringsAsFactors = FALSE, sep = "\t", header = TRUE
)
gtdbtk.ar53_complete_genomes <- read.table(
  "/path/to/gtdbtk/MAG_archaea_complete_genomes/gtdbtk.ar53.summary.tsv",
  stringsAsFactors = FALSE, sep = "\t", header = TRUE
)

gtdbtk.ar53.summary_all <- rbind(
  gtdbtk.ar53_contig,
  gtdbtk.ar53_scaffold,
  gtdbtk.ar53_chromosome,
  gtdbtk.ar53_complete_genomes
)

# Retain only genomes that survived dRep dereplication
dRep <- dir("/path/to/dRep/Thermoproteota/dereplicated_genomes")
dRep_ids <- sapply(dRep, function(x) strsplit(x, ".fna")[[1]][1])
gtdbtk.ar53.summary <- subset(
  gtdbtk.ar53.summary_all,
  user_genome %in% dRep_ids
)

# Class-level taxonomy (starting rank for hierarchical balancing)
gtdbtk.ar53.summary_class <- sapply(
  gtdbtk.ar53.summary$classification,
  function(x) strsplit(x, ";")[[1]][3]
)


##############################
# Stage 2: Exploration & decision phase
#
# Manually inspect taxonomic distributions to decide how many genomes to
# retain at each rank. Results of this phase are recorded in sampling.xlsx.
#
# Procedure at each rank:
#   look at counts → calculate average → keep small lineages fully →
#   drill down into large lineages at the next lower rank
##############################

cat("\n========== Class-level distribution (Thermoproteota) ==========\n")
print(sort(table(gtdbtk.ar53.summary_class), decreasing = TRUE))

# ----- Class: Methanomethylicia -------------------------------------------
cat("\n----- c__Methanomethylicia -----\n")
Methanomethylicia <- gtdbtk.ar53.summary[
  gtdbtk.ar53.summary_class == "c__Methanomethylicia",
]
Methanomethylicia_order <- sapply(
  Methanomethylicia$classification,
  function(x) strsplit(x, ";")[[1]][4]
)
print(sort(table(Methanomethylicia_order), decreasing = TRUE))

# Order B29-G17 → Family DSZF01 → species level
B29_G17 <- Methanomethylicia[Methanomethylicia_order == "o__B29-G17", ]
B29_G17_family <- sapply(B29_G17$classification, function(x) strsplit(x, ";")[[1]][5])
print(sort(table(B29_G17_family), decreasing = TRUE))

DSZF01 <- B29_G17[B29_G17_family == "f__DSZF01", ]
DSZF01_species <- sapply(DSZF01$classification, function(x) strsplit(x, ";")[[1]][7])
print(sort(table(DSZF01_species), decreasing = TRUE))

# Order Nezhaarchaeales
Nezhaarchaeales <- Methanomethylicia[
  Methanomethylicia_order == "o__Nezhaarchaeales",
]
Nezhaarchaeales_family <- sapply(
  Nezhaarchaeales$classification,
  function(x) strsplit(x, ";")[[1]][5]
)
print(sort(table(Nezhaarchaeales_family), decreasing = TRUE))

B40_G2 <- Nezhaarchaeales[Nezhaarchaeales_family == "f__B40-G2", ]
B40_G2_genus <- sapply(B40_G2$classification, function(x) strsplit(x, ";")[[1]][6])
print(sort(table(B40_G2_genus), decreasing = TRUE))

unknown <- B40_G2[B40_G2_genus == "g__", ]
unknown_species <- sapply(unknown$classification, function(x) strsplit(x, ";")[[1]][7])
print(sort(table(unknown_species), decreasing = TRUE))

WYZ_LMO8 <- Nezhaarchaeales[Nezhaarchaeales_family == "f__WYZ-LMO8", ]
WYZ_LMO8_genus <- sapply(WYZ_LMO8$classification, function(x) strsplit(x, ";")[[1]][6])
print(sort(table(WYZ_LMO8_genus), decreasing = TRUE))

WYZ_LMO8_2 <- WYZ_LMO8[WYZ_LMO8_genus == "g__WYZ-LMO8", ]
WYZ_LMO8_2_species <- sapply(
  WYZ_LMO8_2$classification,
  function(x) strsplit(x, ";")[[1]][7]
)
print(sort(table(WYZ_LMO8_2_species), decreasing = TRUE))

# Order Methanomethylicales → Family Methanomethylicaceae → genus → species
Methanomethylicales <- Methanomethylicia[
  Methanomethylicia_order == "o__Methanomethylicales",
]
Methanomethylicales_family <- sapply(
  Methanomethylicales$classification,
  function(x) strsplit(x, ";")[[1]][5]
)
print(sort(table(Methanomethylicales_family), decreasing = TRUE))

Methanomethylicaceae <- Methanomethylicales[
  Methanomethylicales_family == "f__Methanomethylicaceae",
]
Methanomethylicaceae_genus <- sapply(
  Methanomethylicaceae$classification,
  function(x) strsplit(x, ";")[[1]][6]
)
print(sort(table(Methanomethylicaceae_genus), decreasing = TRUE))

for (g in c("g__WYZ-LMO11", "g__Methanomethylicus",
            "g__WYZ-LMO10", "g__Methanosuratincola")) {
  sub <- Methanomethylicaceae[Methanomethylicaceae_genus == g, ]
  sp  <- sapply(sub$classification, function(x) strsplit(x, ";")[[1]][7])
  cat("\nSpecies within ", g, ":\n", sep = "")
  print(sort(table(sp), decreasing = TRUE))
}

# ----- Class: Nitrososphaeria_A -------------------------------------------
cat("\n----- c__Nitrososphaeria_A -----\n")
Nitrososphaeria_A <- gtdbtk.ar53.summary[
  gtdbtk.ar53.summary_class == "c__Nitrososphaeria_A",
]
Nitrososphaeria_A_family <- sapply(
  Nitrososphaeria_A$classification,
  function(x) strsplit(x, ";")[[1]][5]
)
print(sort(table(Nitrososphaeria_A_family), decreasing = TRUE))
# Continue hierarchical inspection for Caldarchaeaceae, HR02,
# Wolframiiraptoraceae and their genera/species as needed.
# Pattern: inspect counts → decide keep-all or drill down one rank.

# ----- Class: Thermoprotei ------------------------------------------------
cat("\n----- c__Thermoprotei -----\n")
Thermoprotei <- gtdbtk.ar53.summary[
  gtdbtk.ar53.summary_class == "c__Thermoprotei",
]
Thermoprotei_order <- sapply(
  Thermoprotei$classification,
  function(x) strsplit(x, ";")[[1]][4]
)
print(sort(table(Thermoprotei_order), decreasing = TRUE))
# Continue hierarchical inspection for large orders.

# ----- Class: Bathyarchaeia -----------------------------------------------
cat("\n----- c__Bathyarchaeia -----\n")
Bathyarchaeia <- gtdbtk.ar53.summary[
  gtdbtk.ar53.summary_class == "c__Bathyarchaeia",
]
Bathyarchaeia_order <- sapply(
  Bathyarchaeia$classification,
  function(x) strsplit(x, ";")[[1]][4]
)
print(sort(table(Bathyarchaeia_order), decreasing = TRUE))
# Large orders (e.g. EX4484-135, B25, B24, RBG-16-48-13, TCS64) are further
# inspected at family → genus → species.

# ----- Class: Nitrososphaeria ---------------------------------------------
cat("\n----- c__Nitrososphaeria -----\n")
Nitrososphaeria <- gtdbtk.ar53.summary[
  gtdbtk.ar53.summary_class == "c__Nitrososphaeria",
]
Nitrososphaeria_order <- sapply(
  Nitrososphaeria$classification,
  function(x) strsplit(x, ";")[[1]][4]
)
print(sort(table(Nitrososphaeria_order), decreasing = TRUE))
# Conexivisphaerales and Nitrososphaerales further broken down as needed.

# After completing inspections for all major classes, record the final
# retention numbers for each terminal lineage in sampling.xlsx.
#
# Expected format of sampling.xlsx (no header row):
#   column 1 : class
#   column 2 : order   (NA if sampling is performed only at class level)
#   column 3 : family  (NA if sampling stops at order level)
#   column 4 : genus   (NA if sampling stops at family level)
#   column 5 : species (NA if sampling stops at genus level)
#   column 6 : n_retain (integer — number of genomes to keep)


##############################
# Stage 3: Execution phase
# Read the pre-decided sampling table and retain the prescribed genomes
##############################

# Pre-compute taxonomy vectors for efficient matching
class_gtdbtk   <- sapply(gtdbtk.ar53.summary$classification, function(x) strsplit(x, ";")[[1]][3])
order_gtdbtk   <- sapply(gtdbtk.ar53.summary$classification, function(x) strsplit(x, ";")[[1]][4])
family_gtdbtk  <- sapply(gtdbtk.ar53.summary$classification, function(x) strsplit(x, ";")[[1]][5])
genus_gtdbtk   <- sapply(gtdbtk.ar53.summary$classification, function(x) strsplit(x, ";")[[1]][6])
species_gtdbtk <- sapply(gtdbtk.ar53.summary$classification, function(x) strsplit(x, ";")[[1]][7])

# Prefer readxl (xlsx depends on Java and is more fragile)
if (!requireNamespace("readxl", quietly = TRUE)) {
  stop("Please install the 'readxl' package: install.packages('readxl')")
}
library(readxl)

# Sheet index corresponds to the Thermoproteota decision table
# (adjust sheet number if sampling.xlsx uses a different layout)
sampling <- read_excel("sampling.xlsx", sheet = 5, col_names = FALSE)
sampling[[6]] <- as.integer(sampling[[6]])

# Select genomes according to the taxonomic depth specified in each row
select_genomes <- function(row) {
  n_levels <- sum(!is.na(unlist(row[1:5])))
  n_keep   <- as.integer(row[[6]])

  if (n_levels == 1) {
    pool <- gtdbtk.ar53.summary$user_genome[class_gtdbtk == row[[1]]]
  } else if (n_levels == 2) {
    pool <- gtdbtk.ar53.summary$user_genome[
      class_gtdbtk == row[[1]] & order_gtdbtk == row[[2]]
    ]
  } else if (n_levels == 3) {
    pool <- gtdbtk.ar53.summary$user_genome[
      class_gtdbtk == row[[1]] & order_gtdbtk == row[[2]] &
        family_gtdbtk == row[[3]]
    ]
  } else if (n_levels == 4) {
    pool <- gtdbtk.ar53.summary$user_genome[
      class_gtdbtk == row[[1]] & order_gtdbtk == row[[2]] &
        family_gtdbtk == row[[3]] & genus_gtdbtk == row[[4]]
    ]
  } else if (n_levels == 5) {
    pool <- gtdbtk.ar53.summary$user_genome[
      class_gtdbtk == row[[1]] & order_gtdbtk == row[[2]] &
        family_gtdbtk == row[[3]] & genus_gtdbtk == row[[4]] &
        species_gtdbtk == row[[5]]
    ]
  } else {
    stop("Invalid number of taxonomic levels in sampling row")
  }

  if (length(pool) <= n_keep) {
    return(pool)
  }
  # Note: Methods describe representative selection prioritising genome
  # quality and within-lineage diversity. The sampling.xlsx table records
  # the final retention numbers; random sampling is used here when the
  # pool exceeds n_keep. For strict reproducibility, replace sample()
  # with a deterministic ranking (e.g. by completeness, contamination, N50).
  sample(pool, n_keep)
}

selected <- unlist(lapply(seq_len(nrow(sampling)), function(i) {
  message("Processing row ", i, " / ", nrow(sampling))
  select_genomes(sampling[i, ])
}))

# Copy selected genomes to the output directory for GS-Present-B
output_dir <- "/path/to/genome_all/Thermoproteota_MAGs_selected/"
dir.create(output_dir, showWarnings = FALSE, recursive = TRUE)

source_dir <- "/path/to/dRep/Thermoproteota/dereplicated_genomes/"

file.copy(
  from = paste0(source_dir, selected, ".fna"),
  to   = output_dir,
  overwrite = FALSE
)

message("Selected ", length(selected), " Thermoproteota genomes for GS-Present-B.")
message("Files written to: ", output_dir)

################################################################################
# End of script
################################################################################
```


## B3 — Four independently curated marker sets (PMSs)

### Construction (Extended Data Fig. 7)
| Dataset | MAG inclusion | PMS | Markers |
| --- | --- | --- | --- |
| D1 | Complete isolates only | PMS-Isolate | 35 |
| D2 | + high-quality MAGs (MIMAG) | PMS-HighMAG1 | 34 |
| D3 | + complete MAGs | PMS-HighMAG2 | 32 |
| D4 | + medium-quality MAGs (CheckM ≥70% / ≤10%) | PMS-MediumMAG | 30 |

MAG sequences were decontaminated before marker selection. Each family was examined in a **single-protein tree**; HGT-like or anomalous domain-level topologies were excluded before concatenation. The four PMSs share **28** core markers but are **not strictly nested**.

**Robustness criterion:** consistent recovery of the same topology across these independently screened sets—despite different composition and MAG-inclusion stringency—is positive evidence for reliability (not an assumption of correctness).


## B4 — Factorial maximum-likelihood phylogenomics (Fig. 2)

### Design
Every genome collection × every PMS under **LG+C60+F+G**, 1,000 ultrafast bootstrap replicates. Alignment: MAFFT-linsi → BMGE (BLOSUM30) → concatenate.

| Panels | Condition |
| --- | --- |
| Fig. 2a–d | GS-Zhang2025-raw × 4 PMSs (contamination present; sampling imbalanced) |
| Fig. 2e–h | GS-Zhang2025-B-raw × 4 PMSs (contamination present; sampling balanced) |
| Fig. 2i–l | GS-Zhang2025-clean × 4 PMSs (decontaminated; sampling imbalanced) |
| Fig. 2m–p | GS-Zhang2025-B-clean × 4 PMSs (**full control**) |

### Distinct effects of the two biases
- **Contamination** → disagreement among PMSs (unstable placements, often high support).  
- **Sampling imbalance** → directional attraction (e.g. recurrent Korarchaeia affinities).  
- **Full control only** → all four PMSs agree: eukaryotes sister to monophyletic TACK–Asgard.

Trees are defined by deposited `.contree` files (`data/trees/maximum_likelihood/`); iTOL/Illustrator are display-only.


### Full script: `scripts/phylogenomics/run_iqtree_ml.sh`

```bash
#!/bin/bash
# ============================================================
# Script : run_iqtree_ml.sh (EXAMPLE TEMPLATE)
# Purpose: Maximum-likelihood phylogenomic inference with
#          IQ-TREE 3 under LG+C60+F+G and ultrafast bootstrap.
#
# Manuscript
# ----------
# Contamination and taxon sampling explain conflicting eukaryote
# placements
#
# Methods mirrored by this script
# --------------------------------
# "Phylogenomic analyses"
#
# For each combination of genome collection (GS) and phylogenetic
# marker set (PMS), single-gene protein alignments were built with
# MAFFT-linsi, trimmed with BMGE (BLOSUM30), concatenated into a
# supermatrix, and analysed in IQ-TREE 3 under LG+C60+F+G. Node
# support was estimated from 1,000 ultrafast bootstrap replicates.
# The factorial design (all GS × all PMS) assesses topological
# stability across contamination control and sampling balance.
#
# Genome collections (examples)
#   GS-Zhang2025 / GS-Zhang2025-B  (raw | clean)
#   GS-Liu2021   / GS-Liu2021-B    (raw | clean)
#   GS-Present-B                   (raw | clean | ultra-clean)
#
# Marker sets
#   PMS-Isolate | PMS-HighMAG1 | PMS-HighMAG2 | PMS-MediumMAG
#
# Tools  : MAFFT, BMGE, catfasta2phyml (or equivalent), IQ-TREE 3
# Stage  : Phylogenomics (primary ML inference)
# Note   : EXAMPLE template. Paths and MPI settings are placeholders.
# ============================================================

set -euo pipefail

# ------------------ Configuration (MODIFY AS NEEDED) ------------------
WORK_DIR="/path/to/your/project"
SOFTWARE_DIR="/path/to/your/software"

# Example: one GS × PMS combination
GS_LABEL="GS-Present-B-clean"
PMS_LABEL="PMS-MediumMAG"

MARKER_FAA_DIR="${WORK_DIR}/data/markers/${GS_LABEL}_${PMS_LABEL}"
# Directory of single-gene FASTA files (one .faa per marker family)

ALIGNMENT_DIR="${WORK_DIR}/results/alignments/${GS_LABEL}_${PMS_LABEL}"
OUTPUT_DIR="${WORK_DIR}/results/trees/maximum_likelihood"
MODEL="LG+C60+F+G"
UFBOOT=1000

THREADS_MAFFT=6
THREADS_IQTREE=48
MPI_MAP="ppr:2:node:PE=48"

mkdir -p "${ALIGNMENT_DIR}" "${OUTPUT_DIR}"

echo "=== IQ-TREE 3 maximum-likelihood analysis ==="
echo "Genome set / PMS : ${GS_LABEL} × ${PMS_LABEL}"
echo "Marker FASTAs    : ${MARKER_FAA_DIR}"
echo "Alignment dir    : ${ALIGNMENT_DIR}"
echo "Tree output dir  : ${OUTPUT_DIR}"
echo "Model            : ${MODEL}"
date
echo ""

# ============================================================
# Step 1: Multiple sequence alignment (MAFFT-linsi)
# ============================================================
echo ">>> [1/4] MAFFT-linsi (per marker)..."
shopt -s nullglob
marker_faas=("${MARKER_FAA_DIR}"/*.faa)

if [[ ${#marker_faas[@]} -eq 0 ]]; then
  echo "WARNING: No .faa files in ${MARKER_FAA_DIR}"
else
  for faa in "${marker_faas[@]}"; do
    base=$(basename "${faa}" .faa)
    echo "  → ${base}"
    mafft-linsi --thread "${THREADS_MAFFT}" \
      "${faa}" \
      > "${ALIGNMENT_DIR}/${base}.aln"
  done
fi
echo ""

# ============================================================
# Step 2: Trim ambiguously aligned regions (BMGE, BLOSUM30)
# ============================================================
echo ">>> [2/4] BMGE trimming (BLOSUM30)..."
aln_files=("${ALIGNMENT_DIR}"/*.aln)

if [[ ${#aln_files[@]} -eq 0 ]]; then
  echo "WARNING: No .aln files in ${ALIGNMENT_DIR}"
else
  for aln in "${aln_files[@]}"; do
    base=$(basename "${aln}" .aln)
    echo "  → ${base}"
    java -Xmx5G -jar "${SOFTWARE_DIR}/BMGE-1.12/BMGE.jar" \
      -i "${aln}" \
      -t AA \
      -m BLOSUM30 \
      -of "${ALIGNMENT_DIR}/${base}.trimmed.aln"
  done
fi
echo ""

# ============================================================
# Step 3: Concatenate trimmed single-gene alignments
# ============================================================
echo ">>> [3/4] Concatenating into supermatrix..."
# catfasta2phyml.pl (or an equivalent concatenator) builds the
# amino-acid supermatrix used as IQ-TREE input.
SUPERMATRIX="${ALIGNMENT_DIR}/${GS_LABEL}_${PMS_LABEL}.faa"

"${SOFTWARE_DIR}/catfasta2phyml.pl" \
  -f "${ALIGNMENT_DIR}"/*.trimmed.aln \
  --concatenate \
  > "${SUPERMATRIX}"

echo "    Supermatrix: ${SUPERMATRIX}"
echo ""

# ============================================================
# Step 4: Maximum-likelihood inference (IQ-TREE 3)
# ============================================================
echo ">>> [4/4] IQ-TREE 3 (${MODEL}, UFBoot=${UFBOOT})..."
PREFIX="${OUTPUT_DIR}/${GS_LABEL}_${PMS_LABEL}"

mpirun --bind-to core --map-by "${MPI_MAP}" \
  "${SOFTWARE_DIR}/iqtree3-mpi" \
  -s "${SUPERMATRIX}" \
  -st AA \
  -m "${MODEL}" \
  -bb "${UFBOOT}" \
  -pre "${PREFIX}" \
  -nt "${THREADS_IQTREE}"

echo "    Tree prefix: ${PREFIX}"
echo ""

# ============================================================
# Completion
# ============================================================
echo "=== IQ-TREE 3 ML analysis finished ==="
date
echo "Key settings used in the study:"
echo "  Model              : LG+C60+F+G"
echo "  Support            : 1,000 ultrafast bootstrap replicates"
echo "  Alignment          : MAFFT-linsi"
echo "  Trimming           : BMGE (BLOSUM30)"
echo "  Design             : all genome collections × all four PMSs"
echo ""
echo "Primary topology under full control of contamination and sampling"
echo "imbalance: eukaryotes sister to a monophyletic TACK–Asgard clade."
echo "See Methods: 'Phylogenomic analyses'."
```


## B5 — Independent tests of the full-control topology (Fig. 3)

| Analysis | Finding | Display |
| --- | --- | --- |
| Post-hoc multi-evidence audit | Extra contig removal **&lt;0.8%** per group; topology unchanged | Fig. 3a |
| Expanded Asgard sampling | Placement unchanged | Fig. 3b |
| PMSF (LG+C60+F+G+PMSF) | Same overall relationship | Fig. 3c |
| AU tests | Supported topology *P* = 0.999; Heimdall/Njord/Hod/TACK alternatives rejected | Fig. 3d |
| CAT-GTR, 10 chains | **7/10** recover TACK–Asgard + eukaryotes; non-convergence → descriptive only | Fig. 3e |

**Primary inference:** contamination-controlled ML + AU tests. Bayesian CAT-GTR is a sensitivity analysis, not posterior consensus support.


### B5a — Independent post-hoc audit

Applied to already-clean collections. Does not rebuild from raw data or change sampling structure. Tools: **GUNC** (bacterial/chimeric ∩ GC anomaly), **VirSorter2 + CheckV** (free virus only; proviruses retained), **Whokaryote** (eukaryotic ∩ GC anomaly). GC anomaly supports bacterial and eukaryotic calls only, not viral decisions.


### Full script: `scripts/independent_audit/independent_audit.sh`

```bash
#!/bin/bash
# ============================================================
# Script : independent_audit.sh (EXAMPLE TEMPLATE)
# Purpose: Independent post-hoc contamination audit on already
#          decontaminated genome collections (e.g. GS-Present-B-
#          clean / GS-Zhang2025-B-clean).
#
# Manuscript
# ----------
# Contamination and taxon sampling explain conflicting eukaryote
# placements
#
# Methods mirrored by this script
# --------------------------------
# "Independent post-hoc contamination audit"
#
# This is a robustness verification step applied after primary
# decontamination. It does not rebuild collections from raw data,
# does not alter taxonomic sampling structure, and does not introduce
# new analysis variables. Residual bacterial, free-virus and
# eukaryotic signals are assessed with source-specific tools under a
# conservative multi-evidence framework. Proviruses were retained by
# default. In practice, additional candidate contamination identified
# at this stage accounted for <0.8% of contigs in each audited
# archaeal group and did not alter the recovered topology
# (eukaryotes sister to a monophyletic TACK–Asgard clade).
#
# Tool roles
# ----------
#   GUNC        bacterial chimerism / taxonomic inconsistency
#               (flag when chimeric AND contig assigned as bacterial;
#                intersected with GC-anomaly contigs)
#   VirSorter2  viral signal detection (score ≥ 0.5; full predictions)
#   CheckV      intersect with VirSorter2; free virus vs provirus
#   Whokaryote  eukaryotic sequence detection
#               (intersected with GC-anomaly contigs)
#   GC anomaly  supports bacterial and eukaryotic calls only
#               (not applied to viral decisions)
#
# Companion R script
# ------------------
#   scripts/independent_audit/merge_audit_flags_and_extract_markers.R
#   Merges primary flags (CAT + geNomad free-virus) with audit flags
#   (GUNC∩GC, VirSorter2∩CheckV, Whokaryote∩GC) and extracts cleaned
#   marker proteins.
#
# Stage  : Robustness (post-hoc audit)
# Note   : EXAMPLE template. Paths are placeholders.
# ============================================================

set -euo pipefail

# ------------------ Configuration (MODIFY AS NEEDED) ------------------
WORK_DIR="/path/to/your/project"
SOFTWARE_DIR="/path/to/your/software"
SCRIPT_DIR="${WORK_DIR}/scripts/independent_audit"

# Already primary-decontaminated genomes (clean collections)
INPUT_GENOMES="${WORK_DIR}/data/genomes/GS-Present-B-clean"

OUTPUT_DIR="${WORK_DIR}/results/independent_audit"
LOG_DIR="${OUTPUT_DIR}/logs"

GUNC_DB="${SOFTWARE_DIR}/gunc_db_gtdb214_2/gunc_db_gtdb214.dmnd"
VIRSORTER_DB="${SOFTWARE_DIR}/VirSorter2/db"

THREADS_GUNC=96
THREADS_VIRSORTER=18
THREADS_CHECKV=4

mkdir -p \
  "${OUTPUT_DIR}/gunc" \
  "${OUTPUT_DIR}/virsorter2" \
  "${OUTPUT_DIR}/checkv" \
  "${OUTPUT_DIR}/whokaryote" \
  "${OUTPUT_DIR}/flags" \
  "${LOG_DIR}"

echo "=== Independent post-hoc contamination audit ==="
echo "Input genomes (clean) : ${INPUT_GENOMES}"
echo "Output directory      : ${OUTPUT_DIR}"
date
echo ""

# ============================================================
# 1. GUNC — bacterial chimerism and taxonomic inconsistency
# ============================================================
# A contig is treated as bacterial-derived contamination when:
#   (1) GUNC_pass indicates chimeric, AND
#   (2) GUNC_contig_bac is Yes
# Downstream, GUNC hits are intersected with GC-anomaly contigs.
echo ">>> GUNC (chimerism / bacterial assignment)..."
gunc run \
  --input_dir "${INPUT_GENOMES}" \
  --file_suffix .fna \
  --threads "${THREADS_GUNC}" \
  -r "${GUNC_DB}" \
  --out_dir "${OUTPUT_DIR}/gunc" \
  --contig_taxonomy_output \
  > "${LOG_DIR}/gunc.log" 2>&1

echo "    GUNC finished."
echo ""

# ============================================================
# 2. VirSorter2 + CheckV — free viral sequences only
# ============================================================
# VirSorter2 provides viral-signal evidence (score ≥ 0.5).
# Only "full" predictions are retained; sequence names are stripped
# of the ||full suffix before intersecting with CheckV contig IDs.
# CheckV classifies structure:
#   free_virus      → candidate for removal
#   provirus        → retained by default
#   host_dominated  → viral signal unreliable; not used for removal
echo ">>> VirSorter2 (viral signal)..."
virsorter run \
  -w "${OUTPUT_DIR}/virsorter2" \
  -i "${INPUT_GENOMES}" \
  --include-groups dsDNAphage,NCLDV,RNA,ssDNA,lavidaviridae \
  -j "${THREADS_VIRSORTER}" \
  --db-dir "${VIRSORTER_DB}" \
  all \
  > "${LOG_DIR}/virsorter2.log" 2>&1

echo ">>> CheckV (free virus vs provirus)..."
checkv end_to_end \
  "${INPUT_GENOMES}" \
  "${OUTPUT_DIR}/checkv" \
  -t "${THREADS_CHECKV}" \
  > "${LOG_DIR}/checkv.log" 2>&1

echo "    VirSorter2 + CheckV finished."
echo ""

# ============================================================
# 3. Whokaryote — eukaryotic contigs
# ============================================================
# Downstream, Whokaryote hits are intersected with GC-anomaly contigs.
echo ">>> Whokaryote (eukaryotic sequence detection)..."
whokaryote.py \
  --contigs "${INPUT_GENOMES}" \
  --outdir "${OUTPUT_DIR}/whokaryote" \
  --model S \
  > "${LOG_DIR}/whokaryote.log" 2>&1

echo "    Whokaryote finished."
echo ""

# ============================================================
# 4. Merge multi-evidence flags and extract cleaned markers (R)
# ============================================================
# Per genome, contigs are excluded if flagged by any of:
#   • geNomad free-virus (topology != Provirus)   [primary]
#   • CAT exogenous contigs                       [primary]
#   • GUNC bacterial/chimeric ∩ GC anomaly        [audit]
#   • VirSorter2 full hits ∩ CheckV               [audit]
#   • Whokaryote eukaryotic ∩ GC anomaly          [audit]
#
# GC anomaly supports bacterial and eukaryotic calls only (not viral).
# Proviruses are retained. The union of the above contig sets is removed
# before orthologue selection.
#
# Companion script:
#   merge_audit_flags_and_extract_markers.R
#
echo ">>> Merging audit flags and extracting markers..."
# Rscript "${SCRIPT_DIR}/merge_audit_flags_and_extract_markers.R"
# Edit paths inside the R script to point at:
#   - tool outputs under ${OUTPUT_DIR}
#   - primary CAT / geNomad flag tables
#   - eggNOG annotation and protein FASTA inventories

echo ""
echo "=== Independent post-hoc audit finished ==="
date
echo "Tool outputs:"
echo "  GUNC       : ${OUTPUT_DIR}/gunc/"
echo "  VirSorter2 : ${OUTPUT_DIR}/virsorter2/"
echo "  CheckV     : ${OUTPUT_DIR}/checkv/"
echo "  Whokaryote : ${OUTPUT_DIR}/whokaryote/"
echo ""
echo "Marker FASTAs after audit: produced by"
echo "  merge_audit_flags_and_extract_markers.R"
echo "Additional contig removal at this stage was <0.8% per archaeal group"
echo "and did not alter the main phylogenetic topology (Methods / Results)."
echo "Primary conclusions remain based on the clean collections; the audit"
echo "verifies that residual contamination is not topology-determining."
```


### Full script: `scripts/independent_audit/merge_audit_flags_and_extract_markers.R`

```r
################################################################################
# Independent post-hoc audit: merge multi-evidence flags and extract markers
#
# Manuscript
# ----------
# Contamination and taxon sampling explain conflicting eukaryote placements
#
# Methods
# -------
# After primary decontamination (CAT + geNomad), residual contamination was
# re-assessed with GUNC (bacterial chimerism), VirSorter2 + CheckV (free virus
# only), and Whokaryote (eukaryotic signal). GC anomaly was used only to
# support bacterial and eukaryotic calls. Contigs flagged by any of these
# independent sources (union) were excluded before orthologue selection.
# Proviruses (geNomad topology == "Provirus") were retained.
#
# Contig sets merged per genome
# -----------------------------
#   • geNomad free-virus          (topology != "Provirus")
#   • CAT exogenous contigs       (primary decontamination flags)
#   • GUNC bacterial/chimeric     (intersected with GC-anomaly contigs)
#   • VirSorter2 full viral hits  (intersected with CheckV contigs)
#   • Whokaryote eukaryotic       (intersected with GC-anomaly contigs)
#
# Input tables are placeholders; adapt paths to the local environment.
################################################################################

library(Biostrings)

if (!requireNamespace("readxl", quietly = TRUE)) {
  stop("Install readxl: install.packages('readxl')")
}
library(readxl)

##############################
# 1. Primary-decontamination flags (still applied)
##############################
genomad_contigs <- read_excel("/path/to/genomad_conservative_MAG_archaea_contig.xlsx", 1)
genomad_scaffolds <- read_excel("/path/to/genomad_conservative_MAG_archaea_scaffold.xlsx", 1)
genomad_chromosomes <- read_excel("/path/to/genomad_conservative_MAG_archaea_chromosome.xlsx", 1)
genomad_complete_genomes <- read.table(
  "/path/to/genomad_conservative_MAG_archaea_complete_genome.txt",
  stringsAsFactors = FALSE, header = TRUE
)
genomad_PRJNA1162170 <- read_excel("/path/to/genomad_conservative_PRJNA1162170.xlsx", 1)
genomad_Zhang_public_Asgard_add <- read_excel(
  "/path/to/genomad_conservative_Zhang_public_Asgard_add.xlsx", 1
)

genomad <- rbind(
  genomad_contigs, genomad_scaffolds, genomad_chromosomes,
  genomad_complete_genomes, genomad_PRJNA1162170, genomad_Zhang_public_Asgard_add
)

cat_Asgardarchaeota <- read.table(
  "/path/to/cat_Asgardarchaeota.txt", stringsAsFactors = FALSE, sep = "\t", header = TRUE
)
cat_PRJNA1162170 <- read.table(
  "/path/to/cat_PRJNA1162170.txt", stringsAsFactors = FALSE, sep = "\t", header = TRUE
)
cat_Zhang_public_Asgard_add <- read.table(
  "/path/to/cat_Zhang_public_Asgard_add.txt", stringsAsFactors = FALSE, sep = "\t", header = TRUE
)
cat_list <- rbind(cat_Asgardarchaeota, cat_PRJNA1162170, cat_Zhang_public_Asgard_add)

##############################
# 2. Post-hoc audit tools
##############################
# GUNC + GC anomaly (GC supports bacterial calls only)
gunc <- read_excel(
  "/path/to/gunc_MAGs_add_PRJNA1162170_public_remove_viruses_and_aberrant_contigs.xlsx", 1
)
gc <- read_excel(
  "/path/to/gc_MAGs_add_PRJNA1162170_public_remove_viruses_and_aberrant_contigs.xlsx", 1
)
gunc$genome_contig <- paste(gunc$genome, gunc$contig, sep = "__")
gc$genome_contig   <- paste(gc$genome, gc$contig, sep = "__")
gunc <- subset(gunc, genome_contig %in% gc$genome_contig)

# VirSorter2 (full contigs only) ∩ CheckV
virsorter <- read_excel(
  "/path/to/virsorter_MAGs_add_PRJNA1162170_public_remove_viruses_and_aberrant_contigs.xlsx", 1
)
checkv <- read_excel(
  "/path/to/checkv_MAGs_add_PRJNA1162170_public_remove_viruses_and_aberrant_contigs.xlsx", 1
)
# Keep VirSorter2 "full" predictions; strip ||full suffix
virsorter <- virsorter[
  sapply(virsorter$seqname, function(o) strsplit(o, "\\|\\|")[[1]][2]) == "full",
]
virsorter$seqname <- sapply(virsorter$seqname, function(p) strsplit(p, "\\|\\|")[[1]][1])
virsorter$genome_contig <- paste(virsorter$genome, virsorter$seqname, sep = "__")
checkv$genome_contig    <- paste(checkv$genome, checkv$contig_id, sep = "__")
virsorter <- subset(virsorter, genome_contig %in% checkv$genome_contig)

# Whokaryote ∩ GC anomaly (GC supports eukaryotic calls only)
whokaryote <- read_excel(
  "/path/to/whokaryote_MAGs_add_PRJNA1162170_public_remove_viruses_and_aberrant_contigs.xlsx", 1
)
whokaryote$genome_contig <- paste(whokaryote$genome, whokaryote$contig, sep = "__")
whokaryote <- subset(whokaryote, genome_contig %in% gc$genome_contig)

##############################
# 3. Genome / annotation inventories
##############################
prodigal <- read.table(
  "/path/to/dereplicated_Asgardarchaeota_add_PRJNA1162170_public_prodigal.txt",
  stringsAsFactors = FALSE, sep = "\t"
)
prodigal_name <- sapply(prodigal$V1, function(u) {
  bn <- strsplit(u, "/")[[1]]
  sub("\\.faa$", "", bn[length(bn)])
})

eggnog_remove_paralogs <- read_excel("/path/to/eggnog.xlsx", sheet = 21)

annotations_file <- dir(pattern = "\\.emapper\\.annotations$")
exclude_annotations <- c(
  "GCF_008000775.2_ASM800077v2_genomic.emapper.annotations",
  "GCA_025839675.1_ASM2583967v1_genomic.emapper.annotations",
  "Heimdallarchaeota_archaeon_HC1.emapper.annotations",
  "Heimdallarchaeota_archaeon_SC1.emapper.annotations"
)
target_annotations <- setdiff(
  intersect(annotations_file, paste0(prodigal_name, ".emapper.annotations")),
  exclude_annotations
)

out_dir <- "/path/to/emapper_faa/.../MAGs_add_PRJNA1162170_public_remove_viruses_and_aberrant_contigs_audit/Asgard_dRep/"
dir.create(out_dir, showWarnings = FALSE, recursive = TRUE)

protein_to_contig <- function(pid) {
  parts <- strsplit(pid, "_")[[1]]
  paste(parts[1:(length(parts) - 1)], collapse = "_")
}

##############################
# 4. Per-genome: union of all flags → drop proteins → extract markers
##############################
for (ann_file in target_annotations) {
  genome_id <- sub("\\.emapper\\.annotations$", "", ann_file)
  message("Audit processing: ", genome_id)

  annotations <- read.delim(
    ann_file, stringsAsFactors = FALSE, comment.char = "#", header = FALSE
  )

  genomad_sub    <- subset(genomad, genome == genome_id)
  cat_sub        <- subset(cat_list, genome == genome_id)
  gunc_sub       <- subset(gunc, genome == genome_id)
  virsorter_sub  <- subset(virsorter, genome == genome_id)
  whokaryote_sub <- subset(whokaryote, genome == genome_id)

  # Union of primary + post-hoc audit flags
  vc_contigs <- unique(c(
    subset(genomad_sub, topology != "Provirus")$seq_name,  # free virus only
    cat_sub[["X..contig"]],
    gunc_sub$contig,
    virsorter_sub$seqname,
    whokaryote_sub$contig
  ))

  protein_contigs <- sapply(annotations$V1, protein_to_contig)
  annotations <- annotations[!(protein_contigs %in% vc_contigs), ]

  faa_path <- prodigal$V1[which(prodigal_name == genome_id)]
  if (length(faa_path) != 1) {
    warning("Protein FASTA missing/ambiguous for ", genome_id)
    next
  }
  protein <- readAAStringSet(faa_path)
  protein_name <- sapply(names(protein), function(v) strsplit(v, " ")[[1]][1])

  for (y in seq_len(nrow(eggnog_remove_paralogs))) {
    root_id <- as.character(eggnog_remove_paralogs$root[y])

    hit <- annotations[
      sapply(annotations$V5, function(z) root_id %in% strsplit(z, ",")[[1]]),
    ]
    if (nrow(hit) == 0) next

    hit_single <- hit[
      sapply(hit$V5, function(z) {
        ranks <- sapply(strsplit(z, ",")[[1]], function(u) strsplit(u, "@")[[1]][2])
        sum(ranks == "1|root") == 1
      }),
    ]
    if (nrow(hit_single) == 0) next

    allowed_second <- strsplit(as.character(eggnog_remove_paralogs[y, "All"]), ";")[[1]]
    allowed_second <- allowed_second[
      sapply(allowed_second, function(t) strsplit(t, "\\|")[[1]][2]) %in%
        c("Archaea", "Eukaryota")
    ]

    hit_second <- hit_single[
      sapply(hit_single$V5, function(w) strsplit(w, ",")[[1]][2]) %in% allowed_second,
    ]
    if (nrow(hit_second) == 0) next

    best_id <- hit_second$V1[order(hit_second$V3)[1]]
    faa_one <- protein[match(best_id, protein_name)]
    names(faa_one) <- genome_id

    family_tag <- strsplit(root_id, "@")[[1]][1]
    writeXStringSet(
      faa_one,
      file.path(out_dir, paste0(family_tag, ".faa")),
      format = "fasta",
      append = TRUE
    )
  }
}

message("Independent audit marker extraction finished.")
message("Additional contig removal at this stage was minor (<0.8% per group)")
message("and did not alter the main phylogenetic topology in the manuscript.")
################################################################################
```


### B5b — Approximately Unbiased (AU) topology tests

Constraint topologies evaluated under the same LG+C60+F+G model. Supported: `((TACK, Asgard), Eukaryotes)`.


### Full script: `scripts/phylogenomics/run_au_topology_tests.sh`

```bash
#!/bin/bash
# ============================================================
# Script : run_au_topology_tests.sh (EXAMPLE TEMPLATE)
# Purpose: Approximately Unbiased (AU) tests of competing
#          eukaryotic-placement topologies in IQ-TREE 3.
#
# Manuscript
# ----------
# Contamination and taxon sampling explain conflicting eukaryote
# placements
#
# Methods mirrored by this script
# --------------------------------
# "Topology testing and Bayesian inference"
#
# Competing topologies (e.g. eukaryotes sister to Korarchaeia,
# Njordarchaeia, other Asgard subgroups, or to a monophyletic
# TACK–Asgard clade) were evaluated with the approximately
# unbiased (AU) test under the same site-heterogeneous model
# used for primary ML inference (LG+C60+F+G). Constraint trees
# were supplied as a multiphyly list (-z); site-wise log-
# likelihoods were resampled (-zb) and AU / other topology-test
# statistics were reported (-au -zw).
#
# Tools  : IQ-TREE 3
# Stage  : Phylogenomics (topology tests)
# Note   : EXAMPLE template. Paths are placeholders.
# ============================================================

set -euo pipefail

# ------------------ Configuration (MODIFY AS NEEDED) ------------------
WORK_DIR="/path/to/your/project"
SOFTWARE_DIR="/path/to/your/software"

# Supermatrix for the GS × PMS combination under test
# (typically a full-control clean + balanced collection)
ALIGNMENT="${WORK_DIR}/data/alignments/GS-Present-B-clean_PMS-MediumMAG.faa"

# Multiphyly file of constraint topologies to compare
# (one Newick tree per line; same taxon set as the alignment)
TREE_LIST="${WORK_DIR}/results/trees/AU_tests/all_constraint_topologies.treels"

OUTPUT_DIR="${WORK_DIR}/results/trees/AU_tests"
MODEL="LG+C60+F+G"
REPS=10000          # RELL replicates for AU (-zb)
THREADS=64

mkdir -p "${OUTPUT_DIR}"

echo "=== Approximately Unbiased (AU) topology tests ==="
echo "Alignment     : ${ALIGNMENT}"
echo "Tree list (-z): ${TREE_LIST}"
echo "Model         : ${MODEL}"
echo "RELL replicates: ${REPS}"
date
echo ""

# ============================================================
# AU tests in IQ-TREE 3
# ============================================================
# -z   : evaluate user trees (constraint / candidate topologies)
# -zb  : number of RELL bootstrap replicates
# -zw  : print weighted topology-test statistics
# -au  : compute the approximately unbiased test
# -n 0 : optional; skip tree search and only score supplied trees
#        (use when -z trees are complete and no ML search is needed)
#
echo ">>> Running AU tests..."
iqtree3 \
  -s "${ALIGNMENT}" \
  -st AA \
  -z "${TREE_LIST}" \
  -zb "${REPS}" \
  -zw \
  -au \
  -n 0 \
  -m "${MODEL}" \
  -nt "${THREADS}" \
  -pre "${OUTPUT_DIR}/AU_test_results"

echo ""
echo "=== AU topology tests finished ==="
date
echo "Results prefix: ${OUTPUT_DIR}/AU_test_results"
echo "Inspect *.iqtree for AU / bp-RELL / KH / SH / ELW statistics."
echo "In the study, topologies placing eukaryotes as sister to a"
echo "monophyletic TACK–Asgard clade were supported under full"
echo "control of contamination and sampling imbalance; alternative"
echo "placements (e.g. sister to Korarchaeia or Njordarchaeia) were"
echo "rejected or not significantly supported depending on the dataset."
echo "See Methods: 'Topology testing and Bayesian inference'."
```


### Full script: `scripts/figure_reproduction/fig3d_au_topology_test.py`

```python
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Fig. 3d — Topology tests (Approximately Unbiased test)

Manuscript
----------
Contamination and taxon sampling explain conflicting eukaryote placements

Reproduces the published panel summarizing AU tests of alternative
eukaryotic placements relative to the supported topology
((TACK, Asgard), Eukaryotes).

Methods / Results context
-------------------------
Competing topologies were evaluated with the approximately unbiased (AU)
test in IQ-TREE under LG+C60+F+G. Under full control of contamination and
taxonomic sampling imbalance, the topology placing eukaryotes as sister to
a monophyletic TACK–Asgard clade was supported. Alternative placements
(e.g. sister to Heimdallarchaeia, Njordarchaeia, Hodarchaeales, or TACK
alone) received low AU P-values and were rejected at the conventional
α = 0.05 threshold.

Data source (deposited with the manuscript)
-------------------------------------------
    data/trees/AU_tests/AU_test_results.iqtree
    and/or robustness summary tables associated with AU topology tests
P-values below are taken from the deposited AU output and are embedded
here for figure layout only.

Dependencies
------------
    pip install matplotlib numpy

Outputs (current working directory)
-----------------------------------
    au_topology_test.png
    au_topology_test.svg
    au_topology_test.pdf

Usage
-----
    python scripts/figure_reproduction/fig3d_au_topology_test.py
"""

import math
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, Circle

mpl.rcParams["font.family"] = "DejaVu Sans"
mpl.rcParams["svg.fonttype"] = "none"
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["axes.linewidth"] = 1.4

# ---------------------------------------------------------------------------
# AU test results (from IQ-TREE AU output / deposited summary tables)
# P = Approximately Unbiased test P-value
# Rejection strength = -log10(P); dashed line = P = 0.05
# ---------------------------------------------------------------------------
supported_topology = "((TACK, Asgard), Eukaryotes)"
supported_p = 0.999

# Alternative constrained topologies and their AU P-values
alternatives = [
    ("(Heimdallarchaeia, Eukaryotes)", 6.36e-05),
    ("(Njordarchaeia, Eukaryotes)", 0.000263),
    ("(Hodarchaeales, Eukaryotes)", 0.000255),
    ("(TACK, Eukaryotes)", 0.00158),
]

labels = [x[0] for x in alternatives]
pvals = np.array([x[1] for x in alternatives], dtype=float)
strength = -np.log10(pvals)
threshold_x = -math.log10(0.05)  # P = 0.05

# ---------------------------------------------------------------------------
# Figure
# ---------------------------------------------------------------------------
fig = plt.figure(figsize=(10.6, 7.2), dpi=300)
ax = fig.add_axes([0.42, 0.15, 0.38, 0.53])

fig.text(0.015, 0.94, "d", fontsize=33, fontweight="bold", va="center")
fig.text(
    0.065, 0.94, "Topology tests (Approximately Unbiased test)",
    fontsize=28, fontweight="bold", va="center",
)

# Supported topology banner
box = FancyBboxPatch(
    (0.055, 0.735), 0.90, 0.135,
    transform=fig.transFigure,
    boxstyle="round,pad=0.004,rounding_size=0.007",
    facecolor="#EEF8EF", edgecolor="#2AAA5A", linewidth=1.5,
)
fig.add_artist(box)

cx, cy, r = 0.11, 0.802, 0.037
fig.add_artist(Circle(
    (cx, cy), r, transform=fig.transFigure,
    facecolor="white", edgecolor="#08A719", linewidth=4,
))
fig.lines.append(mpl.lines.Line2D(
    [cx - 0.019, cx - 0.006, cx + 0.024],
    [cy - 0.002, cy - 0.022, cy + 0.017],
    transform=fig.transFigure, color="#08A719", linewidth=4,
    solid_capstyle="round", solid_joinstyle="round",
))
fig.text(0.165, 0.802, supported_topology, fontsize=27, fontweight="bold", va="center")
fig.text(
    0.77, 0.802, rf"$P = {supported_p:.3f}$",
    fontsize=25, fontweight="bold", style="italic", va="center",
)

# Alternative topologies: lollipop plot of -log10(P)
y = np.arange(len(labels))[::-1]
ax.set_xlim(0, 5)
ax.set_ylim(-0.4, len(labels) - 0.6)

ax.axvline(threshold_x, color="black", linewidth=2.8, linestyle=(0, (5, 4)), zorder=1)
ax.text(
    threshold_x, len(labels) - 0.52, r"$P = 0.05$",
    ha="center", va="bottom", fontsize=16, fontweight="bold", style="italic",
)

line_color = "#E34234"
for yy, xx in zip(y, strength):
    ax.hlines(yy, 0, xx, color=line_color, linewidth=3.2, zorder=2)
    ax.scatter(xx, yy, s=150, color=line_color, zorder=3)

ax.set_yticks(y)
ax.set_yticklabels(labels, fontsize=16, fontweight="bold")

for yy, p in zip(y, pvals):
    if p < 0.001:
        ptxt = f"P = {p:.2e}".replace("e-0", "e−").replace("e-", "e−")
    else:
        ptxt = f"P = {p:.6g}"
    ax.text(
        5.20, yy, ptxt, fontsize=16, fontweight="bold",
        style="italic", ha="left", va="center", clip_on=False,
    )

ax.set_xticks(np.arange(0, 6, 1))
ax.tick_params(axis="x", labelsize=16, width=2.3, length=7)
ax.tick_params(axis="y", width=0, length=0)
for tick in ax.get_xticklabels():
    tick.set_fontweight("bold")

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_linewidth(2.4)
ax.spines["bottom"].set_linewidth(2.4)
ax.set_xlabel(
    r"Rejection strength for alternative topologies, $-\log_{10}(P)$",
    fontsize=16, fontweight="bold", labelpad=10,
)

plt.savefig("au_topology_test.png", dpi=600, bbox_inches="tight", facecolor="white")
plt.savefig("au_topology_test.svg", bbox_inches="tight", facecolor="white")
plt.savefig("au_topology_test.pdf", bbox_inches="tight", facecolor="white")
print("Wrote au_topology_test.{png,svg,pdf}")
```


### B5c — Bayesian CAT-GTR sensitivity analysis

Ten independent PhyloBayes MPI chains. Full convergence was not achieved (expected for this deep divergence under complex site-heterogeneous models). Chain-level topologies are summarised descriptively (**7/10** support TACK–Asgard + eukaryotes).


### Full script: `scripts/phylogenomics/run_phylobayes.sh`

```bash
#!/bin/bash
# ============================================================
# Script : run_phylobayes.sh (EXAMPLE TEMPLATE)
# Purpose: Bayesian phylogenetic inference with PhyloBayes MPI
#          under CAT-GTR, used as a conservative sensitivity
#          analysis relative to primary ML inference.
#
# Manuscript
# ----------
# Contamination and taxon sampling explain conflicting eukaryote
# placements
#
# Methods mirrored by this script
# --------------------------------
# "Topology testing and Bayesian inference"
#
# Bayesian inference under CAT-GTR was performed in PhyloBayes MPI
# on contamination-controlled, taxonomically balanced datasets.
# Ten independent Markov chains were run. Although full
# convergence was not achieved — reflecting the difficulty of
# resolving extremely ancient divergences under complex
# site-heterogeneous models — seven of the ten chains recovered
# topologies in which eukaryotes were placed as sister to a
# monophyletic TACK–Asgard clade, consistent with ML and AU
# results. Remaining chains placed eukaryotes deeper within
# Asgard but did not converge on any single previously proposed
# extant Asgard lineage as the immediate sister group.
#
# All chains used the same burn-in and sampling settings.
# Because convergence diagnostics indicated non-convergence,
# chain-level topologies are summarized descriptively and are
# not interpreted as posterior consensus support. Primary
# phylogenetic inference is based on contamination-controlled
# maximum-likelihood analyses and topology tests.
#
# Tools  : PhyloBayes MPI (pb_mpi, bpcomp, tracecomp)
# Stage  : Phylogenomics (Bayesian sensitivity analysis)
# Note   : EXAMPLE template. Paths and MPI settings are placeholders.
# ============================================================

set -euo pipefail

# ------------------ Configuration (MODIFY AS NEEDED) ------------------
WORK_DIR="/path/to/your/project"
SOFTWARE_DIR="/path/to/your/software"

# Supermatrix for a full-control collection (example)
ALIGNMENT="${WORK_DIR}/data/alignments/GS-Present-B-clean_PMS-MediumMAG.phy"
# PhyloBayes typically expects PHYLIP; convert from FASTA if needed

OUTPUT_DIR="${WORK_DIR}/results/trees/CAT-GTR"
CHAIN_PREFIX="CAT_GTR"
N_CHAINS=10
MPI_NP=54

mkdir -p "${OUTPUT_DIR}"

echo "=== PhyloBayes MPI (CAT-GTR) sensitivity analysis ==="
echo "Alignment     : ${ALIGNMENT}"
echo "Output dir    : ${OUTPUT_DIR}"
echo "Model         : CAT-GTR (-cat -gtr)"
echo "Independent chains: ${N_CHAINS}"
date
echo ""

# ============================================================
# Step 1: Launch independent Markov chains
# ============================================================
# In the study, ten chains were run. Chains may be submitted as
# separate jobs on a cluster rather than sequentially as below.
echo ">>> [1/3] Starting ${N_CHAINS} independent CAT-GTR chains..."

for i in $(seq 1 "${N_CHAINS}"); do
  echo "    Chain ${i}..."
  # Background launch example; prefer one job per chain on HPC:
  mpirun -np "${MPI_NP}" "${SOFTWARE_DIR}/pb_mpi" \
    -d "${ALIGNMENT}" \
    -cat -gtr \
    "${OUTPUT_DIR}/${CHAIN_PREFIX}_chain${i}" \
    > "${OUTPUT_DIR}/${CHAIN_PREFIX}_chain${i}.launch.log" 2>&1 &
done
wait
echo "    All chains launched (or finished, if run synchronously)."
echo ""

# ============================================================
# Step 2: Convergence diagnostics (tracecomp)
# ============================================================
# Compare continuous parameters across chains. In this study,
# diagnostics indicated non-convergence; results were therefore
# not treated as a converged posterior sample.
echo ">>> [2/3] tracecomp (parameter convergence)..."
# Default burn-in is often 1/5 or user-specified; adjust -x as needed.
# Example:
# tracecomp -x 1000 \
#   "${OUTPUT_DIR}/${CHAIN_PREFIX}_chain"{1..10}
echo "    (Inspect effective sizes and rel_diff; non-convergence expected"
echo "     for this deep divergence under CAT-GTR — see Methods.)"
echo ""

# ============================================================
# Step 3: Bipartition comparison (bpcomp) — descriptive only
# ============================================================
# bpcomp compares bipartition frequencies among chains and can
# build a consensus. Because chains did not fully converge,
# topologies are reported at the chain level (e.g. 7/10 supporting
# TACK–Asgard + eukaryotes) rather than as posterior probabilities.
echo ">>> [3/3] bpcomp (bipartition frequencies across chains)..."
bpcomp -c 0.5 \
  -o "${OUTPUT_DIR}/${CHAIN_PREFIX}_bpcomp" \
  "${OUTPUT_DIR}/${CHAIN_PREFIX}_chain1" \
  "${OUTPUT_DIR}/${CHAIN_PREFIX}_chain2" \
  "${OUTPUT_DIR}/${CHAIN_PREFIX}_chain3" \
  "${OUTPUT_DIR}/${CHAIN_PREFIX}_chain4" \
  "${OUTPUT_DIR}/${CHAIN_PREFIX}_chain5" \
  "${OUTPUT_DIR}/${CHAIN_PREFIX}_chain6" \
  "${OUTPUT_DIR}/${CHAIN_PREFIX}_chain7" \
  "${OUTPUT_DIR}/${CHAIN_PREFIX}_chain8" \
  "${OUTPUT_DIR}/${CHAIN_PREFIX}_chain9" \
  "${OUTPUT_DIR}/${CHAIN_PREFIX}_chain10"

echo ""
echo "=== PhyloBayes CAT-GTR analysis finished ==="
date
echo "Chain outputs : ${OUTPUT_DIR}/${CHAIN_PREFIX}_chain*"
echo "bpcomp prefix : ${OUTPUT_DIR}/${CHAIN_PREFIX}_bpcomp"
echo ""
echo "Interpretation (as in the manuscript):"
echo "  • 10 independent chains; full convergence not achieved"
echo "  • 7/10 chains: eukaryotes sister to monophyletic TACK–Asgard"
echo "  • Remaining chains: deeper Asgard placements, no single"
echo "    extant Asgard lineage consistently preferred"
echo "  • Chain-level topologies summarised descriptively only"
echo "  • Primary inference: contamination-controlled ML + AU tests"
echo "See Methods: 'Topology testing and Bayesian inference'."
```


---
# Part C — ESP inventories

**Question:** Does contamination also reshape inferred ESP repertoires?

### Logic
1. Detect ESPs/iESPs (Köstlbacher et al. reference) with DIAMOND + HMMER.  
2. Compare presence before vs after decontamination.  
3. Report lost vs reduced counts by archaeal group (**Fig. 4**).

Asgard contributes the largest number of contamination-sensitive ESPs. Most ESPs remain after cleaning—the claim is selective inflation of some inventories, not that Asgard ESPs are broadly artefactual.


### Full script: `scripts/ESP/run_ESP_pipeline.sh`

```bash
#!/usr/bin/env bash
# ============================================================
# Script : run_ESP_pipeline.sh (EXAMPLE TEMPLATE)
# Purpose: Detect canonical ESPs and extended iESPs in archaeal
#          genomes, enabling comparison before versus after
#          decontamination.
#
# Manuscript
# ----------
# Contamination and taxon sampling explain conflicting eukaryote
# placements
#
# Methods mirrored by this script
# --------------------------------
# ESP/iESP detection was based on the curated family list of
# Köstlbacher et al. For each family, a reference sequence cluster
# was compiled and used both for DIAMOND screening and for building
# a family-specific HMM. Predicted proteins from each genome were
# first screened with DIAMOND (ultra-sensitive); candidate hits were
# then confirmed with HMMER. High-confidence assignments required
# support from both searches. Contaminant-contig filtering
# (CAT / geNomad; primary decontamination) and before/after genome-
# count summaries were performed downstream of this pipeline.
#
# Workflow
# --------
#   1. Build DIAMOND database from ESP/iESP reference sequences
#   2. DIAMOND screen per genome → filter by e-value and coverage
#   3. hmmbuild one HMM per family alignment
#   4. Concatenate family HMMs and hmmpress
#   5. hmmsearch per genome
#
# Tools  : DIAMOND, HMMER (hmmbuild, hmmpress, hmmsearch)
# Note   : EXAMPLE template. Paths are placeholders.
# ============================================================

set -euo pipefail

########################
# Paths (edit these)
########################
WORKDIR="/path/to/your/project"
PROTEIN_DIR="${WORKDIR}/proteins"              # one .faa per genome (Prodigal)
ESP_REF_FAA="${WORKDIR}/reference/ESP_reference.faa"
                                               # merged ESP/iESP family sequences
                                               # headers should retain family IDs
                                               # e.g. >ESP0001|seq1
ALN_DIR="${WORKDIR}/reference/family_alignments"
                                               # one *.aln per ESP/iESP family
ESP_HMM_DIR="${WORKDIR}/reference/ESP_iESP_hmms"
ESP_HMM_ALL="${WORKDIR}/reference/all_families.hmm"

THREADS="${THREADS:-16}"
THREADS_HMM="${THREADS_HMM:-32}"

# DIAMOND filter thresholds (Methods)
EVALUE_MAX="1e-5"
QCOV_MIN=50
SCOV_MIN=50

mkdir -p \
  "${WORKDIR}/diamond/per_genome" \
  "${WORKDIR}/hmmsearch/per_genome" \
  "${ESP_HMM_DIR}"

echo "=== ESP / iESP detection pipeline ==="
echo "Protein directory : ${PROTEIN_DIR}"
echo "Reference FASTA   : ${ESP_REF_FAA}"
date
echo ""

########################
# 1. DIAMOND database
########################
echo "[1/5] Building DIAMOND database from ESP/iESP reference..."
diamond makedb \
  --in "${ESP_REF_FAA}" \
  -d "${WORKDIR}/ESP_reference" \
  --threads "${THREADS}"

########################
# 2. DIAMOND screen (per genome)
########################
echo "[2/5] DIAMOND screening (ultra-sensitive; per genome)..."
shopt -s nullglob
faa_files=("${PROTEIN_DIR}"/*.faa)

if [[ ${#faa_files[@]} -eq 0 ]]; then
  echo "WARNING: No .faa files found in ${PROTEIN_DIR}"
else
  for faa in "${faa_files[@]}"; do
    base=$(basename "$faa" .faa)
    echo "  → ${base}"

    diamond blastp \
      -q "$faa" \
      -d "${WORKDIR}/ESP_reference.dmnd" \
      -o "${WORKDIR}/diamond/per_genome/${base}_vs_ESP.tsv" \
      --ultra-sensitive \
      --evalue "${EVALUE_MAX}" \
      --max-target-seqs 50 \
      --threads "${THREADS}" \
      --outfmt 6 qseqid sseqid pident length qlen slen qcovhsp scovhsp evalue bitscore

    # Keep hits with e-value ≤ 1e-5 and query/subject coverage ≥ 50%
    awk -v emax="${EVALUE_MAX}" -v qmin="${QCOV_MIN}" -v smin="${SCOV_MIN}" \
      'BEGIN{FS=OFS="\t"} $9<=emax+0 && $7>=qmin && $8>=smin' \
      "${WORKDIR}/diamond/per_genome/${base}_vs_ESP.tsv" \
      > "${WORKDIR}/diamond/per_genome/${base}_vs_ESP.filtered.tsv"
  done

  # Optional merged table for downstream summary in R
  cat "${WORKDIR}/diamond/per_genome"/*_vs_ESP.filtered.tsv \
    > "${WORKDIR}/diamond/all_vs_ESP.filtered.tsv" 2>/dev/null || true
fi

########################
# 3. Per-family HMMs from alignments
########################
echo "[3/5] hmmbuild (one HMM per family alignment)..."
aln_files=("${ALN_DIR}"/*.aln)

if [[ ${#aln_files[@]} -eq 0 ]]; then
  echo "WARNING: No .aln files found in ${ALN_DIR}"
else
  for aln in "${aln_files[@]}"; do
    base=$(basename "$aln" .aln)
    echo "  → ${base}"
    hmmbuild --cpu "${THREADS_HMM}" \
      "${ESP_HMM_DIR}/${base}.hmm" \
      "$aln"
  done
fi

########################
# 4. Combine HMMs + hmmpress
########################
echo "[4/5] Concatenating family HMMs and running hmmpress..."
rm -f "${ESP_HMM_ALL}" "${ESP_HMM_ALL}".h3{m,i,f,p}

find "${ESP_HMM_DIR}" -name "*.hmm" -type f -print0 \
  | xargs -0 cat >> "${ESP_HMM_ALL}"

hmmpress "${ESP_HMM_ALL}"

########################
# 5. hmmsearch (per genome)
########################
echo "[5/5] hmmsearch (per genome)..."
if [[ ${#faa_files[@]} -gt 0 ]]; then
  for faa in "${faa_files[@]}"; do
    base=$(basename "$faa" .faa)
    echo "  → ${base}"

    hmmsearch --cpu "${THREADS}" \
      --tblout "${WORKDIR}/hmmsearch/per_genome/${base}_vs_ESP.tbl" \
      "${ESP_HMM_ALL}" \
      "$faa" \
      > "${WORKDIR}/hmmsearch/per_genome/${base}_vs_ESP.out"
  done
fi

echo ""
echo "=== ESP / iESP detection finished ==="
date
echo "  DIAMOND filtered hits : ${WORKDIR}/diamond/per_genome/"
echo "                         ${WORKDIR}/diamond/all_vs_ESP.filtered.tsv"
echo "  HMMER tblout          : ${WORKDIR}/hmmsearch/per_genome/"
echo ""
echo "Downstream (not in this script):"
echo "  - Map protein hits to contigs"
echo "  - Exclude proteins on contaminant contigs (CAT + geNomad free-virus)"
echo "  - Summarise Genome_Count_Before / After per ESP family"
echo "    (see data/ESP/ and Supplementary Tables 11–18)"
```


---
# Figure reproduction notes (full)

The following is the complete `scripts/figure_reproduction/figure_reproduction_README.md` deposited with the repository. It maps each main-figure panel to deposited data and notes which panels are scripted versus assembled in Prism / iTOL / Illustrator / BioRender.


### Full document: `scripts/figure_reproduction/figure_reproduction_README.md`

```markdown
# Figure reproduction notes

This folder documents how main-figure and selected Extended Data panels were produced for:

**Contamination and taxon sampling explain conflicting eukaryote placements**

**Numerical results are defined by the deposited tables, alignments and tree files.**  
Display-only steps (iTOL, Adobe Illustrator, GraphPad Prism, BioRender) are described so that each panel can be matched to its source data. Pixel-identical layout is not required for verification.

Paths below are relative to the repository root unless noted.  
Archived materials will be deposited in Zenodo; the working repository is [GitHub](https://github.com/tjcadd2020/Asgard-Eukaryote-Phylogenomics-2026).

---

## Scripted panels (Python / R)

| Panel | Script | Primary data source |
| --- | --- | --- |
| Fig. 1a | `fig1a_contamination_frequency.py` | `data/decontamination/assessment/contamination_by_lineage.xlsx` (isolate vs MAG frequencies by group and contaminant category) |
| Fig. 1c | `fig1c_contamination_derived_eukaryote_like_proteins.py` | `data/decontamination/assessment/` (frequencies of MAGs harbouring contaminant-derived proteins with apparent similarity to eukaryotic homologues) |
| Fig. 3d | `fig3d_au_topology_test.py` | `data/trees/AU_tests/` (IQ-TREE AU output; *P*-values used in the panel) |
| Extended Data Fig. 1 (viral call overlap) | `plot_extended_data_fig1_venn.R` | `data/viral_detection_comparison/summary_venn_counts.xlsx` |

**Python dependencies**

```bash
pip install matplotlib numpy
```

**R dependencies (Euler diagrams)**

```r
install.packages(c("eulerr", "readxl"))
```

**Run** (from this directory, or adjust relative paths):

```bash
python fig1a_contamination_frequency.py
python fig1c_contamination_derived_eukaryote_like_proteins.py
python fig3d_au_topology_test.py
Rscript plot_extended_data_fig1_venn.R
```

Scripts may embed summary percentages for layout; values must match the deposited assessment tables and AU outputs.

---

## Manually assembled panels

### Fig. 1b

- **Software:** GraphPad Prism  
- **Data:** `data/decontamination/assessment/` (Asgard order-level contamination frequencies; only lineages with ≥10 MAGs)  
- **Note:** Summary frequencies plotted in Prism. No custom analysis script. Stars mark lineages previously proposed as candidate closest relatives of eukaryotes (e.g. Hodarchaeales, Njordarchaeales), which exhibit elevated contamination across multiple categories.

### Fig. 2 (topology panels)

- **Topology source:** `data/trees/maximum_likelihood/*.contree`  
- **Display:** iTOL, then finalized in Adobe Illustrator (colours, labels, layout)  
- **Inference:** IQ-TREE 3 under LG+C60+F+G; black dots indicate ultrafast bootstrap support ≥80%  
- **Note:** Branching order and support values are defined by the deposited `.contree` files. Illustrator was used only for graphical presentation.

**Panel–dataset correspondence** (factorial design: contamination × sampling balance × four independently curated PMSs):

| Panels | Genome set × condition | Interpretation in manuscript |
| --- | --- | --- |
| Fig. 2a–d | GS-Zhang2025-raw × each PMS | Contamination present; sampling imbalanced — marker-set disagreement |
| Fig. 2e–h | GS-Zhang2025-B-raw × each PMS | Contamination present; sampling balanced — marker-set disagreement persists |
| Fig. 2i–l | GS-Zhang2025-clean × each PMS | Decontaminated; sampling imbalanced — directional attraction (e.g. Korarchaeia) |
| Fig. 2m–p | GS-Zhang2025-B-clean × each PMS | Full control (decontaminated and balanced) — all four PMSs converge |

File names follow `GS-*-raw|clean_*_PMS-*.contree` under `data/trees/maximum_likelihood/`.  
Only under simultaneous control of both biases (Fig. 2m–p) do all four PMSs recover eukaryotes as sister to a monophyletic TACK–Asgard archaeal radiation.

### Fig. 3a–c

#### Fig. 3a

- **Content:** Maximum-likelihood robustness on an independently audited ultra-clean genome collection  
- **Topology source:** IQ-TREE consensus trees from the post-hoc audited (ultra-clean) dataset under `data/trees/maximum_likelihood/` (files matching `*ultra-clean*` and the GS × PMS combination shown)  
- **Supporting tables:** post-hoc audit summaries under `data/decontamination/`; robustness summaries under `data/trees/` when deposited  
- **Display:** iTOL → Adobe Illustrator  
- **Note:** Additional contig removal at the audit stage was <0.8% per archaeal group and did not alter the main topology (eukaryotes sister to the monophyletic TACK–Asgard radiation).

#### Fig. 3b

- **Content:** Maximum-likelihood analyses under expanded archaeal taxon sampling  
- **Topology source:** IQ-TREE consensus trees from expanded-sampling collections (alignments under `data/alignments/`; `.contree` under `data/trees/maximum_likelihood/`)  
- **Display:** iTOL → Adobe Illustrator  

#### Fig. 3c

- **Content:** Maximum-likelihood analysis under the alternative site-heterogeneous PMSF approximation (LG+C60+F+G+PMSF)  
- **Topology source:** `data/trees/PMSF/` (files matching the GS × PMS combination shown)  
- **Display:** iTOL → Adobe Illustrator  

**Note (Fig. 3a–c):** Deposited `.contree` files are authoritative for branching order and nodal support. Illustrator was used only for labelling and layout.

### Fig. 3d

- **Script:** `fig3d_au_topology_test.py` (see table above)  
- **Content:** Approximately unbiased (AU) tests of alternative eukaryotic placements relative to ((TACK, Asgard), Eukaryotes)  
- **Data:** IQ-TREE AU output under `data/trees/AU_tests/`  
- **Note:** Alternative placements (e.g. sister to Heimdallarchaeia, Njordarchaeia, Hodarchaeales, or TACK alone) receive low AU *P*-values and are rejected at α = 0.05 under full-control conditions.

### Fig. 3e

- **Software:** GraphPad Prism  
- **Data:** Bayesian chain-level topology summaries (`data/trees/CAT-GTR/`; robustness summary tables when deposited)  
- **Note:** Ten independent PhyloBayes CAT-GTR chains. Plotted from deposited summary counts (e.g. majority of chains recovering TACK–Asgard + eukaryotes). Because full convergence was not achieved, chain-level topologies are descriptive only and are not interpreted as posterior consensus support. No custom plotting script is required beyond the deposited counts.

### Fig. 4a

- **Software:** GraphPad Prism  
- **Data:** `data/ESP/` before/after decontamination tables (e.g. ESP counts by archaeal group)  
- **Note:** Counts of contamination-sensitive ESPs (lost or reduced after decontamination) by major archaeal group. Asgard contributes the largest number of affected ESPs.

### Fig. 4b–c

- **Software:** GraphPad Prism (quantitative elements); BioRender where schematic icons are used  
- **Data:** `data/ESP/` before/after tables  
- **Note:** Numerical values are taken from deposited ESP tables. Schematic elements (e.g. functional icons) may use BioRender. Panels illustrate inventory-level contamination sensitivity and are **not** used to infer ancestral gain–loss histories.

---

## Extended Data and Supplementary figures (selected)

| Figure | Software | Data / notes |
| --- | --- | --- |
| Extended Data Fig. 1 | R (`plot_extended_data_fig1_venn.R`) | Viral-contig call overlap (geNomad vs Phager) across Asgard, TACK, Euryarchaeota and DPANN; `data/viral_detection_comparison/summary_venn_counts.xlsx` |
| Extended Data Fig. 2 | GraphPad Prism | Candidate exogenous sequence burden (>1% of sequences) in isolate genomes vs MAGs; `data/decontamination/assessment/` |
| Extended Data Figs. 3–4 | GraphPad Prism | Contamination frequency vs N50, quality score, CheckM completeness/contamination; exploratory regressions |
| Extended Data Figs. 5–6 | GraphPad Prism | Asgard class/order contamination frequencies and contaminant-derived eukaryote-like proteins (lineages with ≥10 MAGs) |
| Extended Data Fig. 7 | BioRender (workflow schematic) | Single-protein-tree screening and construction of four independently curated PMSs; marker counts and overlap in `data/PMS/` |

BioRender is used for schematic illustration only; it is not required to regenerate numerical panels. A journal-style attribution (e.g. “Created with BioRender.com”) may be placed in Acknowledgements or Methods if required by the publisher.

---

## Related analysis scripts (repository)

| Script | Role |
| --- | --- |
| `scripts/decontamination/run_decontamination.sh` | Primary decontamination (CAT + geNomad) |
| `scripts/decontamination/run_CAT_for_eukaryote_assigned_proteins.sh` | CAT contigs mode to generate ORF2LCA files for eukaryote-assigned proteins on candidate exogenous contigs |
| `scripts/decontamination/count_eukaryote_like_proteins_from_CAT.R` | Count Eukaryota-classified proteins on contigs previously flagged as bacterial, eukaryotic, viral or chimeric |
| `scripts/independent_audit/independent_audit.sh` | Post-hoc multi-evidence audit (GUNC, VirSorter2, CheckV, Whokaryote) |
| `scripts/phylogenomics/run_iqtree_ml.sh` | MAFFT → BMGE → IQ-TREE 3 (LG+C60+F+G, UFBoot) |
| `scripts/phylogenomics/run_au_topology_tests.sh` | AU tests of competing eukaryotic placements |
| `scripts/phylogenomics/run_phylobayes.sh` | CAT-GTR sensitivity analysis (10 independent chains) |
| `scripts/ESP/run_ESP_pipeline.sh` | DIAMOND + HMMER ESP/iESP detection before and after decontamination |

---

## General principles

1. **Trees:** Always prefer deposited `.contree` / PhyloBayes tree files over illustrated topologies.  
2. **Contamination and ESP counts:** Prefer tables under `data/decontamination/assessment/` and `data/ESP/`.  
3. **Scripts in this folder** reproduce selected bar/lollipop-style panels; multi-panel phylogenies and most Prism figures are documented rather than fully scripted.  
4. File names may use `.xlsx` or `.csv` interchangeably if both formats are deposited; column meanings are described in `data/data_README.md` where provided.  
5. **Primary phylogenetic inference** is contamination-controlled maximum-likelihood analysis plus topology tests. Bayesian CAT-GTR results are supportive sensitivity analyses and are summarized at the chain level only because full convergence was not achieved.  
6. Marker-set agreement across the four independently curated PMSs is used as a robustness criterion under factorial control of contamination and taxonomic sampling imbalance; it is not assumed a priori to guarantee correctness.
```


---
## End-to-end map

```text
Public MAGs (Supplementary Tables 2–6)
        │
        ├─ Part A: CAT + geNomad survey
        │    → data/decontamination/assessment/
        │    → Fig. 1
        │
        ▼
Part B: phylogenomics
  B1  Decontaminate each GS     → ~4–5% contigs removed (Extended Data Table 2)
  B2  Hierarchical balancing    → GS-*-B
  B3  Four independently curated PMSs → 35/34/32/30 markers; 28 shared
  B4  ML factorial design       → Fig. 2 (full control: 4/4 PMS agreement)
  B5  Audit / AU / PMSF / CAT-GTR → Fig. 3
        │
        ▼
Part C: ESP before/after decontamination → Fig. 4
```

### Main phylogenomic result
Under simultaneous control of contamination and taxonomic sampling imbalance, all **12** balanced × decontaminated genome-set–marker-set analyses place eukaryotes as sister to a monophyletic **TACK–Asgard** archaeal radiation, outside all currently sampled Asgard subgroups. Primary inference is based on contamination-controlled maximum-likelihood analyses and topology tests; CAT-GTR results are supportive sensitivity analyses summarised at the chain level because full convergence was not achieved.


In [ ]:
from pathlib import Path

checks = [
    "data/decontamination/assessment",
    "data/decontamination/phylogenomic_sets",
    "data/genome_sets",
    "data/PMS",
    "data/alignments",
    "data/trees",
    "data/ESP",
    "data/viral_detection_comparison",
    "scripts/decontamination/run_decontamination.sh",
    "scripts/decontamination/flag_exogenous_contigs_CAT.R",
    "scripts/decontamination/collect_genomad_virus_summaries.R",
    "scripts/decontamination/remove_exogenous_and_extract_markers.R",
    "scripts/decontamination/run_CAT_for_eukaryote_assigned_proteins.sh",
    "scripts/decontamination/count_eukaryote_like_proteins_from_CAT.R",
    "scripts/balancing/hierarchical_balancing_TACK_GS-Present-B.R",
    "scripts/phylogenomics/run_iqtree_ml.sh",
    "scripts/phylogenomics/run_au_topology_tests.sh",
    "scripts/phylogenomics/run_phylobayes.sh",
    "scripts/independent_audit/independent_audit.sh",
    "scripts/independent_audit/merge_audit_flags_and_extract_markers.R",
    "scripts/ESP/run_ESP_pipeline.sh",
    "scripts/figure_reproduction/fig1a_contamination_frequency.py",
    "scripts/figure_reproduction/fig1c_contamination_derived_eukaryote_like_proteins.py",
    "scripts/figure_reproduction/fig3d_au_topology_test.py",
    "scripts/figure_reproduction/plot_extended_data_fig1_venn.R",
    "scripts/figure_reproduction/figure_reproduction_README.md",
]

print("Presence check (run from repository root):")
for f in checks:
    print(f"  [{'OK' if Path(f).exists() else 'missing':7s}] {f}")
